# Endometrial Cancer (EC) PRS Analysis — All of Us CDR v9

Primary association and validation analysis for the EC PRS, using `prs_scores_final_complete.csv` from `EC_PRS_Extraction.ipynb` (2,386 cases / 269,416 controls, N = 271,802). Covers the pooled PRS association, PRS-only vs PRS+PCs and apparent-bootstrap vs cross-validated AUC, ancestry-stratified analysis and standardisation sensitivity, calibration, functional-form checking, and the incident-case and age/BMI sensitivity analyses.


## Setup

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

project_id = os.environ.get('GOOGLE_PROJECT', '').strip().strip("'\"")
%run /home/jupyter/Var_set_2.ipynb  # Var_set_2.ipynb is provided automatically by the AoU Workbench (unique per user/project)
print(project_id)

In [ ]:
# House style for all figures, shared with the other notebooks that produce visualisations
%run /home/jupyter/dissertation_plot_style.py

In [ ]:
result = subprocess.run(['gsutil','-u',project_id,'cp',
    f'{bucket_output}/prs_scores_final_complete.csv', '/home/jupyter/prs_scores_final_complete.csv'],
    capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {result.returncode}): {result.stderr}")
ana_full = pd.read_csv('/home/jupyter/prs_scores_final_complete.csv')
print(f"Loaded: {len(ana_full):,} rows")
print(f"Cases: {int((ana_full['EC_case']==1).sum()):,}  Controls: {int((ana_full['EC_case']==0).sum()):,}")

pc_cols = sorted([c for c in ana_full.columns if c.startswith('PC') and c[2:].isdigit()], key=lambda x: int(x[2:]))[:10]
print(f"Using PCs: {pc_cols}")

## Primary association model (pooled)

In [ ]:
# Primary model: PRS + 10 PCs, pooled across ancestries
model_df = ana_full.dropna(subset=['PRS_z'] + pc_cols)
X = sm.add_constant(model_df[['PRS_z'] + pc_cols])
y = model_df['EC_case']
logit = sm.Logit(y, X).fit(disp=0)

or_prs = np.exp(logit.params['PRS_z'])
ci = np.exp(logit.conf_int().loc['PRS_z'])
auc = roc_auc_score(y, logit.predict(X))

print(f"N = {len(model_df):,} ({int(y.sum()):,} cases)")
print(f"OR per SD of PRS: {or_prs:.4f} (95% CI {ci[0]:.4f}-{ci[1]:.4f}), p={logit.pvalues['PRS_z']:.2e}")
print(f"AUC (PRS + 10 PCs): {auc:.4f}")

In [ ]:
# Checking for duplicate rows feeding the model (would inflate apparent significance/effect size)
model_df = ana_full.dropna(subset=['PRS_z'] + pc_cols)
print("model_df rows:", len(model_df))
print("Unique #IID in model_df:", model_df['#IID'].nunique())
print("Duplicated #IID count:", model_df['#IID'].duplicated().sum())

print("\nEC_case value counts in model_df:")
print(model_df['EC_case'].value_counts())

print("\nMean PRS_z by case status:")
print(model_df.groupby('EC_case')['PRS_z'].agg(['mean','std','count']))

In [ ]:
# Checking for implausible PRS_z extremes and sanity-check the PCs
print("Case PRS_z distribution (percentiles):")
print(ana_full.loc[ana_full['EC_case']==1, 'PRS_z'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

print("\nControl PRS_z distribution (percentiles):")
print(ana_full.loc[ana_full['EC_case']==0, 'PRS_z'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

for pc in pc_cols:
    print(f"\n{pc}: min={ana_full[pc].min():.3f}, max={ana_full[pc].max():.3f}, nunique={ana_full[pc].nunique()}")

## Ancestry-stratified analysis

In [ ]:
def _bootstrap_auc_ci(model_df, pc_cols, n_boot=1000, seed=42):
    """Apparent-performance bootstrap CI for PRS-only and PRS+PCs AUC within one
    ancestry group: resample participants with replacement, refit both models on
    the resample, score AUC on that same resample. Reported as a 95% percentile
    interval around the apparent AUC -- not a held-out/out-of-sample estimate."""
    rng = np.random.default_rng(seed)
    n = len(model_df)
    auc_prs_boot, auc_full_boot = [], []
    n_failed = 0
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boot = model_df.iloc[idx]
        yb = boot['EC_case']
        if yb.nunique() < 2:
            n_failed += 1
            continue
        Xb_prs = sm.add_constant(boot[['PRS_z_within']])
        Xb_full = sm.add_constant(boot[['PRS_z_within'] + pc_cols])
        try:
            fit_prs = sm.Logit(yb, Xb_prs).fit(disp=0)
            fit_full = sm.Logit(yb, Xb_full).fit(disp=0)
        except Exception:
            n_failed += 1
            continue
        # Skipping bootstrap samples where either model fails to converge
        if not fit_prs.mle_retvals.get('converged', True) or not fit_full.mle_retvals.get('converged', True):
            n_failed += 1
            continue
        auc_prs_boot.append(roc_auc_score(yb, fit_prs.predict(Xb_prs)))
        auc_full_boot.append(roc_auc_score(yb, fit_full.predict(Xb_full)))

    n_success = n_boot - n_failed
    print(f"  bootstrap resamples used: {n_success}/{n_boot} ({n_failed} skipped: degenerate, non-converged, or failed to fit)")
    if n_success < 0.95 * n_boot:
        print(f"  WARNING: only {n_success}/{n_boot} resamples succeeded (<95%) -- "
              f"the percentile CI below may be less reliable due to the high failure rate.")
    lo_prs, hi_prs = np.percentile(auc_prs_boot, [2.5, 97.5])
    lo_full, hi_full = np.percentile(auc_full_boot, [2.5, 97.5])
    return lo_prs, hi_prs, lo_full, hi_full


def analyse_ancestry_group(df, label, pc_cols, bootstrap=True):
    df = df.copy()
    n, nc = len(df), int(df['EC_case'].sum())
    if nc < 30:
        return {'Ancestry': label, 'N': n, 'Cases': nc, 'note': 'too few cases'}

    cm = df.loc[df['EC_case']==0, 'PRS_raw'].mean()
    cs = df.loc[df['EC_case']==0, 'PRS_raw'].std()
    df['PRS_z_within'] = (df['PRS_raw'] - cm) / cs

    model_df = df.dropna(subset=['PRS_z_within'] + pc_cols)
    X = sm.add_constant(model_df[['PRS_z_within'] + pc_cols])
    y = model_df['EC_case']
    logit = sm.Logit(y, X).fit(disp=0)

    orr = np.exp(logit.params['PRS_z_within'])
    ci = np.exp(logit.conf_int().loc['PRS_z_within'])
    p = logit.pvalues['PRS_z_within']

    auc_prs = roc_auc_score(y, sm.Logit(y, sm.add_constant(model_df[['PRS_z_within']])).fit(disp=0)
                             .predict(sm.add_constant(model_df[['PRS_z_within']])))
    auc_full = roc_auc_score(y, logit.predict(X))

    result = {'Ancestry': label, 'N': n, 'Cases': nc, 'Controls': n-nc,
              'OR_per_SD': orr, 'CI_lo': ci[0], 'CI_hi': ci[1], 'p': p,
              'AUC_PRS_only': auc_prs, 'AUC_full': auc_full}

    if bootstrap:
        print(f"{label}:")
        lo_prs, hi_prs, lo_full, hi_full = _bootstrap_auc_ci(model_df, pc_cols)
        result.update({'AUC_PRS_only_lo': lo_prs, 'AUC_PRS_only_hi': hi_prs,
                        'AUC_full_lo': lo_full, 'AUC_full_hi': hi_full})

    return result

anc_labels = {'eur':'European', 'afr':'African', 'amr':'Admixed American',
              'eas':'East Asian', 'sas':'South Asian', 'mid':'Middle Eastern',
              'oth':'Other/unclassified ancestry'}

results = [analyse_ancestry_group(ana_full[ana_full['ancestry_pred']==code], label, pc_cols)
           for code, label in anc_labels.items()]

model_df_all = ana_full.dropna(subset=['PRS_z'] + pc_cols)
X_all = sm.add_constant(model_df_all[['PRS_z'] + pc_cols])
y_all = model_df_all['EC_case']
logit_all = sm.Logit(y_all, X_all).fit(disp=0)
results.append({'Ancestry':'Pooled (all ancestries)', 'N': len(model_df_all), 'Cases': int(y_all.sum()),
                 'Controls': int((y_all==0).sum()), 'OR_per_SD': np.exp(logit_all.params['PRS_z']),
                 'CI_lo': np.exp(logit_all.conf_int().loc['PRS_z'])[0], 'CI_hi': np.exp(logit_all.conf_int().loc['PRS_z'])[1],
                 'p': logit_all.pvalues['PRS_z'], 'AUC_PRS_only': None, 'AUC_full': roc_auc_score(y_all, logit_all.predict(X_all))})

results_df = pd.DataFrame(results)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(results_df.to_string(index=False))

results_df.to_csv('/home/jupyter/prs_ancestry_stratified_final.csv', index=False)
subprocess.run(['gsutil','-u',project_id,'cp','/home/jupyter/prs_ancestry_stratified_final.csv', f'{bucket_output}/prs_ancestry_stratified_final.csv'], capture_output=True, text=True)

In [ ]:
# Standardisation-sensitivity test 
# Same pooled model as above, but with PRS_z recoded to ancestry-standardised units instead of the global (pooled-control) standardisation
# This isolates the effect of the predictor's scale, as distinct from the ancestry-stratified table (which fits a separate model per ancestry group)
ana_scale_test = ana_full.copy()
ana_scale_test['PRS_z_within_ancestry'] = np.nan

for acode in ana_scale_test['ancestry_pred'].dropna().unique():
    mask = ana_scale_test['ancestry_pred'] == acode
    ctrl_mask = mask & (ana_scale_test['EC_case']==0)
    cm = ana_scale_test.loc[ctrl_mask, 'PRS_raw'].mean()
    cs = ana_scale_test.loc[ctrl_mask, 'PRS_raw'].std()
    ana_scale_test.loc[mask, 'PRS_z_within_ancestry'] = (ana_scale_test.loc[mask, 'PRS_raw'] - cm) / cs

model_scale = ana_scale_test.dropna(subset=['PRS_z_within_ancestry'] + pc_cols)
X_scale = sm.add_constant(model_scale[['PRS_z_within_ancestry'] + pc_cols])
y_scale = model_scale['EC_case']
m_scale = sm.Logit(y_scale, X_scale).fit(disp=0)

orr_scale = np.exp(m_scale.params['PRS_z_within_ancestry'])
ci_scale = np.exp(m_scale.conf_int().loc['PRS_z_within_ancestry'])

print(f"=== Pooled model, global PRS_z (baseline, from the primary model above) ===")
print(f"OR: {np.exp(logit_all.params['PRS_z']):.4f} (95% CI "
      f"{np.exp(logit_all.conf_int().loc['PRS_z'])[0]:.4f}-{np.exp(logit_all.conf_int().loc['PRS_z'])[1]:.4f})")

print(f"\n=== Pooled model, ancestry-standardized PRS_z (scale test) ===")
print(f"N={len(y_scale):,}, cases={int(y_scale.sum()):,}")
print(f"OR: {orr_scale:.4f} (95% CI {ci_scale[0]:.4f}-{ci_scale[1]:.4f}), p={m_scale.pvalues['PRS_z_within_ancestry']:.2e}")
print(f"AUC: {roc_auc_score(y_scale, m_scale.predict(X_scale)):.4f}")

print(f"\nNote: this pooled ancestry-standardized OR differs from the per-ancestry ORs in "
      f"the ancestry-stratified table above because those fit a SEPARATE model per ancestry "
      f"group, while this fits ONE pooled model after standardizing the predictor within each "
      f"group -- the two are different quantities, not a consistency check on the same one.")

In [ ]:
# Sensitivity analysis: exclude controls flagged for a prior hyperplasia/EIN diagnosis (a recognised but non-universal EC precursor) 
# Re-fits pooled and ancestry-stratified models to check this doesn't materially change the results.
# hyperplasia_ein_flag is already computed and is reused here
if 'hyperplasia_ein_flag' not in ana_full.columns:
    import pandas_gbq
    CDR = os.environ['WORKSPACE_CDR']
    control_ids = ana_full.loc[ana_full['EC_case']==0, '#IID'].astype(int).tolist()
    patterns = ["%endometrial hyperplasia%", "%hyperplasia of endometrium%",
                "%complex atypical hyperplasia%", "%atypical endometrial hyperplasia%",
                "%endometrial intraepithelial neoplasia%"]
    like_sql = " OR ".join("LOWER(concept_name) LIKE '" + p + "'" for p in patterns)
    concept_query = "SELECT concept_id FROM `" + CDR + ".concept` WHERE domain_id='Condition' AND (" + like_sql + ")"
    concepts = pandas_gbq.read_gbq(concept_query, project_id=project_id, dialect='standard', progress_bar_type=None)
    c_str = ','.join(map(str, concepts['concept_id'].tolist()))

    def chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]
    parts = []
    for b in chunks(control_ids, 40000):
        s = ','.join(map(str, b))
        q = ("SELECT DISTINCT co.person_id FROM `" + CDR + ".condition_occurrence` co "
             "WHERE co.person_id IN (" + s + ") "
             "AND (co.condition_concept_id IN (" + c_str + ") OR co.condition_source_concept_id IN (" + c_str + "))")
        parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
    hyp_ids = set(pd.concat(parts, ignore_index=True)['person_id'].astype(int)) if parts else set()
    ana_full['hyperplasia_ein_flag'] = ana_full['#IID'].isin(hyp_ids)
    print("hyperplasia_ein_flag was missing from the loaded file -- recomputed live from condition_occurrence.")
print(f"Hyperplasia/EIN flagged controls: {int(ana_full['hyperplasia_ein_flag'].sum()):,}")

ana_sens = ana_full[~ana_full['hyperplasia_ein_flag']].copy()

def analyse_group_sens(df, label, pc_cols, use_within=True):
    df = df.copy()
    n, nc = len(df), int(df['EC_case'].sum())
    if nc < 30:
        return {'Ancestry': label, 'N': n, 'Cases': nc, 'note': 'too few cases'}
    if use_within:
        cm = df.loc[df['EC_case']==0, 'PRS_raw'].mean(); cs = df.loc[df['EC_case']==0, 'PRS_raw'].std()
        df['PRS_z_s'] = (df['PRS_raw'] - cm) / cs
    else:
        df['PRS_z_s'] = df['PRS_z']
    model_df = df.dropna(subset=['PRS_z_s'] + pc_cols)
    X = sm.add_constant(model_df[['PRS_z_s'] + pc_cols])
    y = model_df['EC_case']
    logit = sm.Logit(y, X).fit(disp=0)
    orr = np.exp(logit.params['PRS_z_s']); ci = np.exp(logit.conf_int().loc['PRS_z_s'])
    return {'Ancestry': label, 'N': n, 'Cases': nc, 'Controls': n-nc,
            'OR_per_SD': orr, 'CI_lo': ci[0], 'CI_hi': ci[1], 'p': logit.pvalues['PRS_z_s']}

results_sens = [analyse_group_sens(ana_sens, 'Pooled (excl. hyperplasia/EIN)', pc_cols, use_within=False)]
for code, label in anc_labels.items():
    results_sens.append(analyse_group_sens(ana_sens[ana_sens['ancestry_pred']==code], f"{label} (excl.)", pc_cols, use_within=True))

results_sens_df = pd.DataFrame(results_sens)
print(results_sens_df.to_string(index=False))

results_sens_df.to_csv('/home/jupyter/prs_hyperplasia_sensitivity.csv', index=False)
subprocess.run(['gsutil','-u',project_id,'cp','/home/jupyter/prs_hyperplasia_sensitivity.csv', f'{bucket_output}/prs_hyperplasia_sensitivity.csv'], capture_output=True, text=True)

## Incident-case sensitivity analysis

In [ ]:
# Classifying each case as incident (diagnosed on/after AoU consent) or prevalent, using consent date from the observation table.
import pandas_gbq
CDR = os.environ['WORKSPACE_CDR']
case_ids_final = ana_full.loc[ana_full['EC_case']==1, '#IID'].astype(int).tolist()

def chunks(l,n):
    for i in range(0,len(l),n): yield l[i:i+n]

parts = []
for b in chunks(case_ids_final, 40000):
    s = ','.join(map(str, b))
    q = f"""
    SELECT person_id, MIN(observation_date) AS consent_date
    FROM `{CDR}.observation`
    WHERE person_id IN ({s}) AND observation_source_concept_id = 1586099
    GROUP BY person_id
    """
    parts.append(pandas_gbq.read_gbq(q, project_id=project_id, dialect='standard', progress_bar_type=None))
consent = pd.concat(parts, ignore_index=True)
consent['consent_date'] = pd.to_datetime(consent['consent_date'])

ana_full['first_diagnosis_date'] = pd.to_datetime(ana_full['first_diagnosis_date'])
ana_full = ana_full.merge(consent.rename(columns={'person_id':'#IID'}), on='#IID', how='left')
ana_full['days_dx_minus_consent'] = (ana_full['first_diagnosis_date'] - ana_full['consent_date']).dt.days
ana_full['incident_flag'] = ana_full['days_dx_minus_consent'] >= 0
ana_full['incident_flag_buffer365'] = ana_full['days_dx_minus_consent'] >= 365

case_mask = ana_full['EC_case']==1
print(f"Incident (dx >= consent): {int(ana_full.loc[case_mask,'incident_flag'].sum()):,}")
print(f"Incident + 12mo buffer: {int(ana_full.loc[case_mask,'incident_flag_buffer365'].sum()):,}")

# Cross-checking incident_flag against EC_Cohort_Characteristics.ipynb's independently computed incident_status per case
result = subprocess.run(['gsutil','-u',project_id,'cp',
    f'{bucket_output}/03_ec_cohort_with_baseline_chars.csv', '/home/jupyter/03_ec_cohort_with_baseline_chars.csv'],
    capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {result.returncode}): {result.stderr}")
chars_cohort = pd.read_csv('/home/jupyter/03_ec_cohort_with_baseline_chars.csv', low_memory=False)

chars_status = chars_cohort.loc[chars_cohort['EC_case']==1, ['person_id', 'incident_status']].rename(
    columns={'person_id': '#IID'}
)
status_check = ana_full.loc[case_mask, ['#IID', 'incident_flag']].merge(
    chars_status, on='#IID', how='outer', validate='one_to_one', indicator=True
)
assert status_check['_merge'].eq('both').all(), (
    f"Case set mismatch between this notebook's incident classification and "
    f"EC_Cohort_Characteristics' saved incident_status -- "
    f"{(status_check['_merge'] != 'both').sum()} case(s) present in only one of the two."
)
# incident_status is 3-way (adds UNKNOWN); mapping incident_flag onto it rather than treating every False as PREVALENT.
status_check['this_notebook_status'] = np.where(
    status_check['incident_flag'], 'INCIDENT',
    np.where(status_check['incident_status'] == 'UNKNOWN', 'UNKNOWN', 'PREVALENT')
)
mismatches = status_check[status_check['this_notebook_status'] != status_check['incident_status']]
assert len(mismatches) == 0, (
    f"{len(mismatches)} case(s) classified differently between this notebook and "
    f"EC_Cohort_Characteristics' saved incident_status:\n{mismatches[['#IID','this_notebook_status','incident_status']].to_string(index=False)}"
)
print(f"\nIncident/prevalent cross-check vs EC_Cohort_Characteristics: PASSED "
      f"({len(status_check):,} cases, all classifications agree)")

In [ ]:
# Incident-case sensitivity analysis 
# Re-fit the PRS+PCs model on progressively tighter incident-case restrictions
# Most cases are prevalent relative to AoU enrolment, unlike Wang et al.'s incident-only UKB design, so this checks whether including prevalent cases distorts the pooled association
def run_incident_variant(incident_mask_col, label):
    keep = (ana_full['EC_case']==0) | ((ana_full['EC_case']==1) & ana_full[incident_mask_col].fillna(False))
    df = ana_full[keep].copy()
    n, nc = len(df), int(df['EC_case'].sum())

    model_df = df.dropna(subset=['PRS_z'] + pc_cols)
    X = sm.add_constant(model_df[['PRS_z'] + pc_cols])
    y = model_df['EC_case']
    logit = sm.Logit(y, X).fit(disp=0)
    orr = np.exp(logit.params['PRS_z']); ci = np.exp(logit.conf_int().loc['PRS_z'])

    eur = df[df['ancestry_pred']=='eur'].copy()
    cm = eur.loc[eur['EC_case']==0,'PRS_raw'].mean(); cs = eur.loc[eur['EC_case']==0,'PRS_raw'].std()
    eur['PRS_z_w'] = (eur['PRS_raw']-cm)/cs
    model_eur = eur.dropna(subset=['PRS_z_w']+pc_cols)
    Xe = sm.add_constant(model_eur[['PRS_z_w']+pc_cols])
    ye = model_eur['EC_case']
    logit_eur = sm.Logit(ye, Xe).fit(disp=0)
    orr_e = np.exp(logit_eur.params['PRS_z_w']); ci_e = np.exp(logit_eur.conf_int().loc['PRS_z_w'])

    print(f"\n=== {label}: N={n:,}, cases={nc:,} ===")
    print(f"Pooled OR/SD: {orr:.4f} ({ci[0]:.4f}-{ci[1]:.4f}), p={logit.pvalues['PRS_z']:.2e}")
    print(f"EUR-only OR/SD:  {orr_e:.4f} ({ci_e[0]:.4f}-{ci_e[1]:.4f}), EUR cases={int(ye.sum()):,}")
    return {'label':label, 'N':n, 'cases':nc, 'OR_pooled':orr, 'OR_lo':ci[0], 'OR_hi':ci[1],
            'OR_eur':orr_e, 'eur_cases':int(ye.sum())}

# Full-cohort row 
# Computed by reusing the pooled model already fitted above and fitting the same EUR-only specification run_incident_variant() uses
eur_full = ana_full[ana_full['ancestry_pred']=='eur'].copy()
cm_full = eur_full.loc[eur_full['EC_case']==0,'PRS_raw'].mean()
cs_full = eur_full.loc[eur_full['EC_case']==0,'PRS_raw'].std()
eur_full['PRS_z_w'] = (eur_full['PRS_raw']-cm_full)/cs_full
model_eur_full = eur_full.dropna(subset=['PRS_z_w']+pc_cols)
Xe_full = sm.add_constant(model_eur_full[['PRS_z_w']+pc_cols])
ye_full = model_eur_full['EC_case']
logit_eur_full = sm.Logit(ye_full, Xe_full).fit(disp=0)
orr_e_full = np.exp(logit_eur_full.params['PRS_z_w'])

# Full-cohort point estimate/CI recomputed directly from the PRS+10PCs model fitted at the top of the notebook (model_df/X/y/logit)
# This is the PRS-plus-PCs estimate
or_full_row = np.exp(logit.params['PRS_z'])
ci_full_row = np.exp(logit.conf_int().loc['PRS_z'])

r_full = {'label': 'FULL COHORT (prevalent+incident)', 'N': len(model_df), 'cases': int(y.sum()),
          'OR_pooled': or_full_row, 'OR_lo': ci_full_row[0], 'OR_hi': ci_full_row[1],
          'OR_eur': orr_e_full, 'eur_cases': int(ye_full.sum())}

r_incident = run_incident_variant('incident_flag', 'INCIDENT ONLY (dx on/after consent)')
r_buffer = run_incident_variant('incident_flag_buffer365', 'INCIDENT + 12mo BUFFER')

print("\n=== SUMMARY: full cohort vs incident-only vs Wang benchmark ===")
print(f"{'Specification':<35}{'N cases':>10}{'Pooled OR/SD':>15}{'EUR OR/SD':>12}")
print(f"{'Full cohort (prevalent+incident)':<35}{r_full['cases']:>10}{r_full['OR_pooled']:>15.2f}{r_full['OR_eur']:>12.2f}")
print(f"{'Incident only':<35}{r_incident['cases']:>10}{r_incident['OR_pooled']:>15.2f}{r_incident['OR_eur']:>12.2f}")
print(f"{'Incident + 12mo buffer':<35}{r_buffer['cases']:>10}{r_buffer['OR_pooled']:>15.2f}{r_buffer['OR_eur']:>12.2f}")
# Wang et al.'s 982/2.13 is a Cox-model hazard ratio from their own incident-only UKB cohort, not a logistic OR and not fitted on this cohort's EUR subset
# Included for rough visual reference only.
print(f"{'Wang et al. UKB (incident only, HR not OR)':<45}{982:>10}{'2.13 (HR)':>15}{'2.13 (HR)':>12}")

In [ ]:
# Forest plot of the incident-case sensitivity analysis pooled OR under progressively tighter incident-case restrictions
forest_rows = [
    ('Full cohort',              r_full['OR_pooled'],     r_full['OR_lo'],     r_full['OR_hi']),
    ('Incident only',            r_incident['OR_pooled'], r_incident['OR_lo'], r_incident['OR_hi']),
    ('Incident, 12-mo buffer',   r_buffer['OR_pooled'],   r_buffer['OR_lo'],   r_buffer['OR_hi']),
]

fig, ax = plt.subplots(figsize=(5.6, 2.6))
y_pos = np.arange(len(forest_rows))[::-1]  # top row = full cohort

for (label, orr, lo, hi), y in zip(forest_rows, y_pos):
    ax.plot([lo, hi], [y, y], color=NAVY, linewidth=1.8, zorder=1)
    ax.plot([lo, lo], [y - 0.12, y + 0.12], color=NAVY, linewidth=1.8, zorder=1)
    ax.plot([hi, hi], [y - 0.12, y + 0.12], color=NAVY, linewidth=1.8, zorder=1)
    ax.scatter([orr], [y], color=NAVY, s=60, zorder=2)
    ax.text(hi + 0.025, y, f"{orr:.2f} ({lo:.2f}–{hi:.2f})", va='center', fontsize=9, color=NAVY)

ax.axvline(1.0, color=GREY, linestyle='--', linewidth=1.2, zorder=0)
ax.text(1.0, max(y_pos) + 0.55, 'OR = 1', ha='center', fontsize=8.5, color='#666666')

ax.set_yticks(y_pos)
ax.set_yticklabels([r[0] for r in forest_rows])
ax.set_ylim(min(y_pos) - 0.6, max(y_pos) + 0.9)
ax.set_xlabel('Pooled odds ratio per SD of PRS (95% CI)')
style_ax(ax)
plt.tight_layout()
savefig_both(fig, 'incident_sensitivity_forest', project_id, bucket_output)
plt.show()

In [ ]:
# Percentile-bin contrast table, using the middle quintile (40-60%) as reference
# Matches Wang et al.'s UKB design for a descriptive comparison.

df = ana_full.dropna(subset=['PRS_z'] + pc_cols).copy()

# Percentiles computed over all participants (cases + controls), not a controls-only reference distribution
# Noted here since Wang et al.'s own percentile bins may use a controls-only reference, in which case bin membership is not exactly comparable like-for-like
df['prs_pctile'] = df['PRS_z'].rank(pct=True) * 100
print("Percentile basis: ranked over ALL participants (cases + controls), not controls-only.")

def bin_pctile(p):
    if p < 1: return 'Bottom 1%'
    elif p < 10: return '1-10%'
    elif p < 40: return '10-40%'
    elif p < 60: return 'Middle quintile (40-60%)'
    elif p < 90: return '60-90%'
    elif p < 99: return '90-99%'
    else: return 'Top 1%'

df['prs_bin'] = df['prs_pctile'].apply(bin_pctile)
bin_order = ['Bottom 1%', '1-10%', '10-40%', 'Middle quintile (40-60%)', '60-90%', '90-99%', 'Top 1%']
df['prs_bin'] = pd.Categorical(df['prs_bin'], categories=bin_order, ordered=True)

X = sm.add_constant(pd.concat([
    pd.get_dummies(df['prs_bin'], drop_first=False).drop(columns=['Middle quintile (40-60%)']).astype(float),
    df[pc_cols]
], axis=1))
y = df['EC_case']
model = sm.Logit(y, X).fit(disp=0)

results = []
for b in bin_order:
    n = (df['prs_bin']==b).sum()
    cases = df.loc[df['prs_bin']==b, 'EC_case'].sum()
    if b == 'Middle quintile (40-60%)':
        results.append({'Bin': b, 'N': n, 'Cases': cases, 'OR': 1.00, 'CI_low': '-', 'CI_high': '-', 'p': '-'})
    else:
        orr = np.exp(model.params[b])
        ci = np.exp(model.conf_int().loc[b])
        results.append({'Bin': b, 'N': n, 'Cases': cases, 'OR': round(orr,2),
                         'CI_low': round(ci[0],2), 'CI_high': round(ci[1],2), 'p': model.pvalues[b]})

decile_table = pd.DataFrame(results)
print(decile_table.to_string(index=False))

decile_table.to_csv('/home/jupyter/prs_percentile_table.csv', index=False)
subprocess.run(['gsutil','-u',project_id,'cp','/home/jupyter/prs_percentile_table.csv', f'{bucket_output}/prs_percentile_table.csv'], capture_output=True, text=True)

## Histological subtype feasibility check

In [ ]:
# Subtype-specific analysis (endometrioid vs non-endometrioid) was explored as a possibility
# This checks whether AoU has usable pathology/histology concepts for EC cases at all

import pandas_gbq
CDR = os.environ['WORKSPACE_CDR']
result = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/02_ec_final_cohort_excluded.csv', '/home/jupyter/02_ec_final_cohort_excluded.csv'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {result.returncode}): {result.stderr}")
final_cohort = pd.read_csv('/home/jupyter/02_ec_final_cohort_excluded.csv', low_memory=False)
case_ids = final_cohort.loc[final_cohort['EC_case']==1, 'person_id'].astype(int).tolist()

q_concepts = f"""
SELECT DISTINCT concept_id, concept_code, concept_name, vocabulary_id
FROM `{CDR}.concept`
WHERE domain_id = 'Measurement'
  AND (
    LOWER(concept_name) LIKE '%histologic%endometri%'
    OR LOWER(concept_name) LIKE '%histologic grade%endometrial%'
    OR LOWER(concept_name) LIKE '%endometrial cancer specimen%'
    OR LOWER(concept_name) LIKE '%endometrioid%'
    OR (LOWER(concept_name) LIKE '%serous%' AND LOWER(concept_name) LIKE '%endometri%')
    OR (LOWER(concept_name) LIKE '%clear cell%' AND LOWER(concept_name) LIKE '%endometri%')
    OR LOWER(concept_name) LIKE '%carcinosarcoma%'
  )
ORDER BY concept_name
"""
hist_concepts = pandas_gbq.read_gbq(q_concepts, project_id=project_id, dialect='standard', progress_bar_type=None)
print(f"Histology/pathology-related measurement concepts found: {len(hist_concepts)}")

In [ ]:
concept_ids = hist_concepts['concept_id'].tolist()
c_str = ','.join(map(str, concept_ids))

q_coverage = f"""
SELECT m.measurement_concept_id, c.concept_name, COUNT(DISTINCT m.person_id) AS n_people
FROM `{CDR}.measurement` m
JOIN `{CDR}.concept` c ON c.concept_id = m.measurement_concept_id
WHERE m.measurement_concept_id IN ({c_str})
GROUP BY m.measurement_concept_id, c.concept_name
ORDER BY n_people DESC
"""
coverage = pandas_gbq.read_gbq(q_coverage, project_id=project_id, dialect='standard', progress_bar_type=None)
print(f"Concepts with ANY data in the entire AoU CDR (not just our cohort): {len(coverage)} / {len(concept_ids)}")

q_our_cases = f"""
SELECT DISTINCT m.person_id
FROM `{CDR}.measurement` m
WHERE m.measurement_concept_id IN ({c_str})
  AND m.person_id IN ({','.join(map(str, case_ids))})
"""
our_cov = pandas_gbq.read_gbq(q_our_cases, project_id=project_id, dialect='standard', progress_bar_type=None)
print(f"\n>>> Of our {len(case_ids):,} EC cases, how many have ANY pathology-panel measurement: {len(our_cov)}")
print("Conclusion: AoU has the vocabulary for detailed EC histology but no participants have this\n"
      "data populated — subtype-specific analysis is not currently possible with this dataset.")

## Age- and BMI-adjusted models

In [ ]:
# Merging in BMI and age for the age/BMI-adjusted models
# bmi_baseline_date is merged in alongside bmi_baseline (not just the value)
# This is so the BMI-timing check further down can read the exact measurement date the model uses
# This way we avoid re-deriving it with a separate query that could drift out of sync
result = subprocess.run(['gsutil','-u',project_id,'cp',
    f'{bucket_output}/03_ec_cohort_with_baseline_chars.csv', '/home/jupyter/cohort_with_baseline.csv'],
    capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {result.returncode}): {result.stderr}")
cohort_bmi = pd.read_csv('/home/jupyter/cohort_with_baseline.csv', low_memory=False)

merge_cols = ['person_id', 'bmi_baseline', 'bmi_baseline_date', 'bmi_n_measurements', 'bmi_sd', 'age_display']
model_data = ana_full.merge(
    cohort_bmi[merge_cols].rename(columns={'person_id':'#IID'}),
    on='#IID', how='left'
)
print(f"Merged: {len(model_data):,}")
print(f"Missing BMI: {model_data['bmi_baseline'].isna().sum():,}")
print(f"Missing age: {model_data['age_display'].isna().sum():,}")

In [ ]:
# Nested models: PRS+PCs, +age, +age+BMI (full), standardised against controls
ctrl = model_data[model_data['EC_case']==0]
age_m, age_s = ctrl['age_display'].mean(), ctrl['age_display'].std()
bmi_m, bmi_s = ctrl['bmi_baseline'].mean(), ctrl['bmi_baseline'].std()
model_data['age_z'] = (model_data['age_display'] - age_m) / age_s
model_data['bmi_z'] = (model_data['bmi_baseline'] - bmi_m) / bmi_s

d1 = model_data.dropna(subset=['PRS_z'] + pc_cols)
X1 = sm.add_constant(d1[['PRS_z'] + pc_cols]); y1 = d1['EC_case']
m1 = sm.Logit(y1, X1).fit(disp=0); auc1 = roc_auc_score(y1, m1.predict(X1))

d2 = model_data.dropna(subset=['PRS_z', 'age_z'] + pc_cols)
X2 = sm.add_constant(d2[['PRS_z', 'age_z'] + pc_cols]); y2 = d2['EC_case']
m2 = sm.Logit(y2, X2).fit(disp=0); auc2 = roc_auc_score(y2, m2.predict(X2))

d3 = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols)
X3 = sm.add_constant(d3[['PRS_z', 'age_z', 'bmi_z'] + pc_cols]); y3 = d3['EC_case']
m3 = sm.Logit(y3, X3).fit(disp=0); auc3 = roc_auc_score(y3, m3.predict(X3))

def report(m, X, y, auc, label):
    print(f"\n=== {label} (N={len(y):,}, cases={int(y.sum()):,}) ===")
    for var in [c for c in X.columns if c != 'const' and not c.startswith('PC')]:
        orr = np.exp(m.params[var]); ci = np.exp(m.conf_int().loc[var])
        print(f"  {var:<10} OR {orr:.4f} (95% CI {ci[0]:.4f}-{ci[1]:.4f}), p={m.pvalues[var]:.2e}")
    print(f"  AUC: {auc:.4f}")

report(m1, X1, y1, auc1, "Model 1: PRS + 10 PCs")
report(m2, X2, y2, auc2, "Model 2: PRS + age + 10 PCs")
report(m3, X3, y3, auc3, "Model 3: PRS + age + BMI + 10 PCs (FULL)")

print(f"\nAUC progression: {auc1:.4f} -> {auc2:.4f} -> {auc3:.4f}")

In [ ]:
# No-PRS baseline model (age + BMI + PCs only) for the incremental-AUC comparison below
d0 = model_data.dropna(subset=['age_z', 'bmi_z'] + pc_cols)
X0 = sm.add_constant(d0[['age_z', 'bmi_z'] + pc_cols]); y0 = d0['EC_case']
m0 = sm.Logit(y0, X0).fit(disp=0); auc0 = roc_auc_score(y0, m0.predict(X0))

report(m0, X0, y0, auc0, "Model 0: age + BMI + 10 PCs, NO PRS (baseline)")

print(f"\nAUC with no PRS (age + BMI + PCs only): {auc0:.4f}")
print(f"AUC with PRS added (age + BMI + PRS + PCs): {auc3:.4f}")
print(f"Marginal AUC gain from adding PRS on top of age + BMI: {auc3 - auc0:+.4f}")

In [ ]:
# Confirm the full model (d3) and no-PRS baseline (d0) use identical participants
# This means the AUC difference is a genuine paired comparison, and not partly driven by a different complete-case sample between the two specifications
ids_d3 = set(d3['#IID']) if '#IID' in d3.columns else set(d3.index)
ids_d0 = set(d0['#IID']) if '#IID' in d0.columns else set(d0.index)
assert ids_d3 == ids_d0, (
    f"d3 and d0 use different participant sets ({len(ids_d3)} vs {len(ids_d0)}); "
    f"the marginal AUC gain is not a clean paired comparison until this is resolved."
)
print(f"Confirmed: d3 (full model) and d0 (no-PRS baseline) use identical participant sets, N={len(ids_d3):,}.")

## Explained variation (Nagelkerke / liability-scale R²)

In [ ]:
# Nagelkerke pseudo-R^2 and liability-scale R^2 (Lee et al. 2012)
# For comparison against Wang et al.'s reported values. 
# Lee et al. define the liability transformation's input as:
    # the R^2 from an ordinary linear regression of the binary phenotype on the predictor, 
    # not a logistic pseudo-R^2
# These are materially different quantities, so both are computed explicitly below.
from scipy.stats import norm

def nagelkerke_r2(y, X_full, X_null):
    l_full = sm.Logit(y, X_full).fit(disp=0).llf
    l_null = sm.Logit(y, X_null).fit(disp=0).llf
    n = len(y)
    cs = 1 - np.exp((2/n)*(l_null - l_full))
    return cs / (1 - np.exp((2/n)*l_null))

def linear_observed_r2(y, x):
    # Observed-scale R^2 input to the Lee et al. liability transformation:
    # R^2 from an OLS regression of the 0/1 phenotype on the predictor.
    X = sm.add_constant(x)
    ols = sm.OLS(y, X).fit()
    return ols.rsquared

def lee_liability_r2(r2_obs, K, P):
    t = norm.ppf(1-K); z = norm.pdf(t); m = z/K
    C = (K*(1-K)/z**2) * (K*(1-K)/(P*(1-P)))
    theta = m*(P-K)/(1-K) * (m*(P-K)/(1-K) - t)
    return r2_obs * C / (1 + r2_obs*theta*C)

y = d1['EC_case'].reset_index(drop=True)
Xint = sm.add_constant(pd.Series(1.0, index=y.index, name='const').to_frame())
Xprs_only = sm.add_constant(d1[['PRS_z']].reset_index(drop=True))

nk_prs_only = nagelkerke_r2(y, Xprs_only, Xint)
r2_obs_linear = linear_observed_r2(y, d1[['PRS_z']].reset_index(drop=True))
P = y.mean()

print(f"Sample: N={len(y):,}, cases={int(y.sum()):,} ({P*100:.2f}%)")
print(f"\nNagelkerke's R2 (PRS alone, vs intercept): {nk_prs_only*100:.2f}%   [Wang et al: 0.9%]")
print(f"(Logistic pseudo-R2 -- reported for context only; NOT the input to the liability-scale")
print(f" transformation below, since Lee et al. (2012) define that transformation's input as")
print(f" the R2 from a linear regression of the 0/1 phenotype, not a logistic pseudo-R2.)")
print(f"\nLinear observed-scale R2 (PRS alone, OLS on 0/1 phenotype): {r2_obs_linear*100:.4f}%")
print(f"\nLiability-scale R2 (Lee et al. 2012), using the correctly-specified linear observed-scale R2:")
for K in [0.02, 0.03, 0.05]:
    print(f"  K={K:.0%}: {lee_liability_r2(r2_obs_linear, K, P)*100:.2f}%   [Wang et al, K=3%: 1.9%]")
print(f"  CAVEAT: sample case fraction {P*100:.2f}% is far below any assumed population prevalence K,")
print(f"  which substantially inflates this conversion — treat as indicative only, not precise.")

In [ ]:
# Nagelkerke R^2 for PRS's own contribution on top of the 10 PCs (PRS+PCs vs PCs-only)
# Distinct from the PRS-alone-vs-intercept figure above
Xpcs_only = sm.add_constant(d1[pc_cols].reset_index(drop=True))
Xprs_plus_pcs = sm.add_constant(d1[['PRS_z'] + pc_cols].reset_index(drop=True))

nk_prs_plus_pcs_vs_intercept = nagelkerke_r2(y, Xprs_plus_pcs, Xint)
nk_prs_plus_pcs_vs_pcs_only = nagelkerke_r2(y, Xprs_plus_pcs, Xpcs_only)

print(f"Nagelkerke's R2 (PRS alone, vs intercept-only):      {nk_prs_only*100:.2f}%")
print(f"Nagelkerke's R2 (PRS + 10 PCs, vs intercept-only):   {nk_prs_plus_pcs_vs_intercept*100:.2f}%")
print(f"Nagelkerke's R2 (PRS's own contribution, vs PCs-only baseline): {nk_prs_plus_pcs_vs_pcs_only*100:.2f}%")
print(f"\nThese are three distinct quantities -- the dissertation specifies exactly which one")
print(f"it cites and compares to Wang et al.'s reported 0.9%.")

In [ ]:
# Pooled PRS-only AUC (no PCs, no age, no BMI), for direct comparison to Wang et al.'s own reported PRS-only AUC (0.671)
m_prs_only = sm.Logit(y, Xprs_only).fit(disp=0)
auc_prs_only = roc_auc_score(y, m_prs_only.predict(Xprs_only))
print(f"Pooled PRS-only AUC (no PCs): {auc_prs_only:.4f}")
print(f"Pooled PRS-plus-10-PCs AUC (headline, Section 4.2): {auc1:.4f}")
print(f"Wang et al. PRS-only AUC: 0.671")

## Ancestry-standardisation checks (Test (a)-(d))

In [ ]:
# Figure: ancestry-specific CONTROL PRS density under pooled (A) vs within-ancestry (B) standardisation
# Controls only, since this figure's job is the standardisation-reference shift, not case/control separation
import seaborn as sns

major_ancestries = ['eur', 'afr', 'amr', 'eas']
anc_display = {'eur': 'European', 'afr': 'African', 'amr': 'Admixed American', 'eas': 'East Asian'}
anc_color = {'eur': NAVY, 'afr': CORAL, 'amr': TEAL, 'eas': '#6B7280'}
SUPPRESS_EXACT_N = {'eas'}

ctrl_only = ana_full[ana_full['EC_case'] == 0]

fig, axes = plt.subplots(2, 1, figsize=(6.2, 7.2), sharey=False)

# Panel A: pooled (global) standardisation, shared x-axis
ax = axes[0]
for anc in major_ancestries:
    sub = ctrl_only.loc[ctrl_only['ancestry_pred'] == anc, 'PRS_z']
    label = anc_display[anc] if anc not in SUPPRESS_EXACT_N else f"{anc_display[anc]} (N: see Table 7)"
    sns.kdeplot(sub, ax=ax, color=anc_color[anc], linewidth=2.2, label=label)
ax.axvline(0, color='#BBBBBB', linestyle=':', linewidth=1)
ax.set_xlabel('PRS_z (pooled standardisation)')
ax.set_ylabel('Density')
ax.set_ylim(0, ax.get_ylim()[1] * 1.28)
ax.set_title('A. Global standardisation', loc='left', fontsize=11, fontweight='bold')
style_ax(ax)

# Panel B: within-ancestry standardisation
ax = axes[1]
for anc in major_ancestries:
    sub = ctrl_only.loc[ctrl_only['ancestry_pred'] == anc, 'PRS_raw']
    sub_z = (sub - sub.mean()) / sub.std()
    sns.kdeplot(sub_z, ax=ax, color=anc_color[anc], linewidth=2.2, label=anc_display[anc])
ax.axvline(0, color='#BBBBBB', linestyle=':', linewidth=1)
ax.set_xlabel('PRS_z (within-ancestry standardisation)')
ax.set_ylabel('')
ax.set_title('B. Within-ancestry standardisation', loc='left', fontsize=11, fontweight='bold')
style_ax(ax)

# One shared legend for both panels, 
# Built from Panel A's handles since both panels show the same four series in the same colours.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.04),
           ncol=2, frameon=False, fontsize=9)

plt.tight_layout()
savefig_both(fig, 'prs_density_by_ancestry', project_id, bucket_output)
plt.show()

In [ ]:
# Checking whether each ancestry group's own PRS_z distribution is unimodal (vs the pooled distribution's visible multimodality
from scipy import stats as sps

for anc in major_ancestries:
    sub = ana_full[ana_full['ancestry_pred'] == anc]['PRS_z'].dropna()
    print(f"{anc}: N={len(sub):,}, mean={sub.mean():.3f}, std={sub.std():.3f}, skew={sps.skew(sub):.3f}, kurtosis={sps.kurtosis(sub):.3f}")

In [ ]:
# Age-PRS correlation as a QC check
# PRS is genotype-derived and shouldn't correlate with age (genotype doesn't change over a lifetime)
# An association would flag age-dependent selection or cohort-composition effects
result = subprocess.run(['gsutil','-u',project_id,'cp', f'{bucket_output}/03_ec_cohort_with_baseline_chars.csv', '/home/jupyter/cohort_age_qc.csv'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {result.returncode}): {result.stderr}")
cohort_age = pd.read_csv('/home/jupyter/cohort_age_qc.csv', low_memory=False)

age_prs_check = ana_full[['#IID', 'PRS_z', 'EC_case']].merge(
    cohort_age[['person_id', 'age_display']].rename(columns={'person_id':'#IID'}),
    on='#IID', how='inner'
).dropna(subset=['PRS_z', 'age_display'])

print(f"N={len(age_prs_check):,}")
corr = age_prs_check['PRS_z'].corr(age_prs_check['age_display'])
print(f"\nCorrelation between PRS_z and age: {corr:.4f}")

for status, label in [(0,'Controls'), (1,'Cases')]:
    sub = age_prs_check[age_prs_check['EC_case']==status]
    c = sub['PRS_z'].corr(sub['age_display'])
    print(f"  {label}: r={c:.4f} (N={len(sub):,})")

In [ ]:
# Does the pooled age-PRS correlation collapse within each ancestry group? 
# European participants are both the oldest group on average and have the lowest raw PRS scale
# Ancestry composition is a plausible contributor to the pooled correlation above.
age_anc_check = age_prs_check.merge(
    ana_full[['#IID', 'ancestry_pred']], on='#IID', how='left'
)

print("Mean age by ancestry group:")
print(age_anc_check.groupby('ancestry_pred')['age_display'].agg(['mean', 'std', 'count']))

print("\nMean PRS_z by ancestry group (for reference, from the earlier violin plot):")
print(age_anc_check.groupby('ancestry_pred')['PRS_z'].agg(['mean', 'std']))

print("\nAge-PRS correlation WITHIN each ancestry group:")
for anc in age_anc_check['ancestry_pred'].dropna().unique():
    sub = age_anc_check[age_anc_check['ancestry_pred']==anc]
    if len(sub) > 100:
        c = sub['PRS_z'].corr(sub['age_display'])
        print(f"  {anc}: r={c:.4f} (N={len(sub):,})")

In [ ]:
# Does adding ancestry as an explicit covariate to the pooled model change the PRS association
# Alternative to the ancestry-stratified approach used above
ancestry_dummies = pd.get_dummies(ana_full['ancestry_pred'], prefix='anc', drop_first=True).astype(float)
anc_cols = ancestry_dummies.columns.tolist()

model_data_anc = pd.concat([ana_full[['PRS_z', 'EC_case'] + pc_cols].reset_index(drop=True),
                             ancestry_dummies.reset_index(drop=True)], axis=1).dropna()

X_A = sm.add_constant(model_data_anc[['PRS_z'] + pc_cols])
y_A = model_data_anc['EC_case']
m_A = sm.Logit(y_A, X_A).fit(disp=0)
auc_A = roc_auc_score(y_A, m_A.predict(X_A))

X_B = sm.add_constant(model_data_anc[['PRS_z'] + pc_cols + anc_cols])
y_B = model_data_anc['EC_case']
m_B = sm.Logit(y_B, X_B).fit(disp=0)
auc_B = roc_auc_score(y_B, m_B.predict(X_B))

print(f"Model A (PRS + PCs): OR {np.exp(m_A.params['PRS_z']):.4f}, AUC {auc_A:.4f}")
print(f"Model B (PRS + PCs + ancestry): OR {np.exp(m_B.params['PRS_z']):.4f}, AUC {auc_B:.4f}")
print(f"Change: OR {100*(np.exp(m_B.params['PRS_z'])/np.exp(m_A.params['PRS_z'])-1):+.1f}%, AUC {auc_B-auc_A:+.4f}")

In [ ]:
# Ancestry-stratified forest plot
forest_rows = results_df.loc[~results_df['Ancestry'].isin(['South Asian', 'Middle Eastern'])].copy()
forest_rows = forest_rows[forest_rows['OR_per_SD'].notna()].reset_index(drop=True)

# Order: pooled at top, then ancestry groups by descending OR
pooled_row = forest_rows[forest_rows['Ancestry'].str.contains('Pooled')]
group_rows = forest_rows[~forest_rows['Ancestry'].str.contains('Pooled')].sort_values('OR_per_SD', ascending=False)
forest_rows = pd.concat([pooled_row, group_rows], ignore_index=True)

fig, ax = plt.subplots(figsize=(5.3, 3.4))
y_pos = np.arange(len(forest_rows))[::-1]
is_pooled = forest_rows['Ancestry'].str.contains('Pooled')
colors = [NAVY if p else GREY for p in is_pooled]

ax.errorbar(forest_rows['OR_per_SD'], y_pos,
            xerr=[forest_rows['OR_per_SD'] - forest_rows['CI_lo'], forest_rows['CI_hi'] - forest_rows['OR_per_SD']],
            fmt='none', ecolor=GREY, capsize=0, zorder=1)
ax.scatter(forest_rows['OR_per_SD'], y_pos, c=colors, s=60, zorder=2, edgecolor='none')

ax.axvline(1, color=GREY, linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(forest_rows['Ancestry'])
ax.set_xlabel('Odds ratio per PRS SD')

# Directing value labels to the right of each point
x_label_pos = forest_rows['CI_hi'].max() * 1.08
for yi, row in zip(y_pos, forest_rows.itertuples()):
    ax.text(x_label_pos, yi, f"{row.OR_per_SD:.2f} ({row.CI_lo:.2f}-{row.CI_hi:.2f})",
            va='center', ha='left', fontsize=9, color='#333333')

ax.set_xlim(forest_rows['CI_lo'].min()*0.9, x_label_pos*1.35)
style_ax(ax)
plt.tight_layout()
savefig_both(fig, 'prs_forest_ancestry_stratified', project_id, bucket_output)
plt.show()

In [ ]:
# Figure: PRS-only vs PRS+10PCs AUC by ancestry group
# With apparent-performance bootstrap 95% CIs (1000 resamples per group, see _bootstrap_auc_ci above)
# Shown as horizontal error bars around each point
auc_rows = results_df[results_df['AUC_full'].notna() & results_df['AUC_PRS_only'].notna()].copy()
auc_rows = auc_rows.sort_values('AUC_full', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(5.6, 3.8))
y_pos = np.arange(len(auc_rows))
has_ci = 'AUC_PRS_only_lo' in auc_rows.columns and 'AUC_full_lo' in auc_rows.columns

for i, row in auc_rows.iterrows():
    ax.plot([row['AUC_PRS_only'], row['AUC_full']], [i, i], color='#CCCCCC', linewidth=1.5, zorder=1)

if has_ci:
    prs_err = np.abs(np.vstack([auc_rows['AUC_PRS_only'] - auc_rows['AUC_PRS_only_lo'],
                                 auc_rows['AUC_PRS_only_hi'] - auc_rows['AUC_PRS_only']]))
    full_err = np.abs(np.vstack([auc_rows['AUC_full'] - auc_rows['AUC_full_lo'],
                                  auc_rows['AUC_full_hi'] - auc_rows['AUC_full']]))
    ax.errorbar(auc_rows['AUC_PRS_only'], y_pos, xerr=prs_err, fmt='none', ecolor=GREY,
                elinewidth=1.2, capsize=3, zorder=1.5)
    ax.errorbar(auc_rows['AUC_full'], y_pos, xerr=full_err, fmt='none', ecolor=NAVY,
                elinewidth=1.2, capsize=3, zorder=1.5)

ax.scatter(auc_rows['AUC_PRS_only'], y_pos, color=GREY, s=55, zorder=2, label='PRS only')
ax.scatter(auc_rows['AUC_full'], y_pos, color=NAVY, s=55, zorder=2, label='PRS + 10 PCs')

# Widen the left-hand plotting range so PRS-only labels/error bars near AUC=0.50 don't collide with the ancestry y-tick labels
# Widen the right-hand range to leave room for the PRS+PCs upper CI bound and its text label.
if has_ci:
    x_min = min(auc_rows['AUC_PRS_only_lo'].min(), 0.5) - 0.10
    x_max = auc_rows['AUC_full_hi'].max() + 0.11
else:
    x_min = min(auc_rows['AUC_PRS_only'].min(), 0.5) - 0.075
    x_max = auc_rows['AUC_full'].max() + 0.06
ax.set_xlim(x_min, x_max)

# Label placement anchors past each point's own CI whisker (not the point itself)
# A fixed small offset from the point would collide with the whisker cap and, in several ancestry rows, with the opposite series' label
label_pad = 0.012
for i, row in auc_rows.iterrows():
    full_anchor = row['AUC_full_hi'] + label_pad if has_ci else row['AUC_full'] + label_pad
    prs_anchor = row['AUC_PRS_only_lo'] - label_pad if has_ci else row['AUC_PRS_only'] - label_pad
    ax.text(full_anchor, i, f"{row['AUC_full']:.3f}", va='center', fontsize=9, color=NAVY)
    ax.text(prs_anchor, i, f"{row['AUC_PRS_only']:.3f}", va='center', ha='right', fontsize=9, color='#666666')

ax.set_yticks(y_pos)
ax.set_yticklabels(auc_rows['Ancestry'])
ax.axvline(0.5, color='#BBBBBB', linestyle=':', linewidth=1)
ax.set_xlabel('AUC')
style_ax(ax)
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)
plt.tight_layout()
savefig_both(fig, 'prs_auc_by_ancestry_prs_vs_full', project_id, bucket_output)
plt.show()

## Incident-only and BMI-timing sensitivity checks

In [ ]:
# Temporal-design check: re-fit the full model on incident cases only. 
# BMI (measured near AoU consent) may postdate diagnosis for prevalent cases (75.6% of cases)
# This re-fit thus gives an approximately cleaner-ordered check of whether the full-model result holds up
# A small combined subgroup either lacks baseline BMI or has it postdate diagnosis (quantified, with small-cell disclosure handling)
# "Approximately", not universally, ordered

incident_keep = (model_data['EC_case']==0) | ((model_data['EC_case']==1) & model_data['incident_flag'].fillna(False))
md_incident = model_data[incident_keep].copy()
print(f"Incident-only subset: N={len(md_incident):,}, cases={int(md_incident['EC_case'].sum()):,}")

d1i = md_incident.dropna(subset=['PRS_z'] + pc_cols)
X1i = sm.add_constant(d1i[['PRS_z'] + pc_cols]); y1i = d1i['EC_case']
m1i = sm.Logit(y1i, X1i).fit(disp=0); auc1i = roc_auc_score(y1i, m1i.predict(X1i))

d3i = md_incident.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols)
X3i = sm.add_constant(d3i[['PRS_z', 'age_z', 'bmi_z'] + pc_cols]); y3i = d3i['EC_case']
m3i = sm.Logit(y3i, X3i).fit(disp=0); auc3i = roc_auc_score(y3i, m3i.predict(X3i))

d0i = md_incident.dropna(subset=['age_z', 'bmi_z'] + pc_cols)
X0i = sm.add_constant(d0i[['age_z', 'bmi_z'] + pc_cols]); y0i = d0i['EC_case']
m0i = sm.Logit(y0i, X0i).fit(disp=0); auc0i = roc_auc_score(y0i, m0i.predict(X0i))

report(m1i, X1i, y1i, auc1i, "INCIDENT-ONLY Model 1: PRS + 10 PCs")
report(m3i, X3i, y3i, auc3i, "INCIDENT-ONLY Model 3: PRS + age + BMI + 10 PCs (FULL)")
report(m0i, X0i, y0i, auc0i, "INCIDENT-ONLY Model 0: age + BMI + 10 PCs, NO PRS")

print(f"\n=== Temporal-design comparison ===")
print(f"Full cohort,   age+BMI-only AUC: {auc0:.4f}  ->  +PRS AUC: {auc3:.4f}  (gain {auc3-auc0:+.4f})")
print(f"Incident-only, age+BMI-only AUC: {auc0i:.4f}  ->  +PRS AUC: {auc3i:.4f}  (gain {auc3i-auc0i:+.4f})")
print(f"\nFull-cohort full-model PRS OR: {np.exp(m3.params['PRS_z']):.4f}")
print(f"Incident-only full-model PRS OR: {np.exp(m3i.params['PRS_z']):.4f}")

In [ ]:
# Apparent-performance bootstrap CI for the AUC gain from adding PRS
# Resample participants (with replacement)
# Refit both the age+BMI-only and full age+BMI+PRS models on each resample
# Record the AUC difference
np.random.seed(42)
n_boot = 1000
d_full = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols).reset_index(drop=True)
n = len(d_full)
auc_diffs = []

for b in range(n_boot):
    idx = np.random.choice(n, size=n, replace=True)
    boot = d_full.iloc[idx]
    y_b = boot['EC_case']
    if y_b.sum() < 5 or y_b.sum() > len(y_b) - 5:
        continue  # skip degenerate resamples

    X0b = sm.add_constant(boot[['age_z', 'bmi_z'] + pc_cols])
    X3b = sm.add_constant(boot[['PRS_z', 'age_z', 'bmi_z'] + pc_cols])
    try:
        m0b = sm.Logit(y_b, X0b).fit(disp=0)
        m3b = sm.Logit(y_b, X3b).fit(disp=0)
    except Exception:
        continue
    # Skip bootstrap samples where either model fails to converge.
    if not m0b.mle_retvals.get('converged', True) or not m3b.mle_retvals.get('converged', True):
        continue
    auc0b = roc_auc_score(y_b, m0b.predict(X0b))
    auc3b = roc_auc_score(y_b, m3b.predict(X3b))
    auc_diffs.append(auc3b - auc0b)

auc_diffs = np.array(auc_diffs)
print(f"Bootstrap resamples completed: {len(auc_diffs):,} / {n_boot}")
print(f"Observed AUC gain (point estimate): {auc3 - auc0:+.4f}")
print(f"Bootstrap mean AUC gain: {auc_diffs.mean():+.4f}")
ci_lo, ci_hi = np.percentile(auc_diffs, [2.5, 97.5])
print(f"95% bootstrap CI for AUC gain: ({ci_lo:+.4f}, {ci_hi:+.4f})")
print(f"Proportion of bootstrap resamples with AUC gain <= 0: {(auc_diffs <= 0).mean():.3f}")

In [ ]:
# Basic (reflected) bootstrap CI for the AUC gain, as an alternative to the percentile CI above
# C* = [2*theta_hat - q_hi*, 2*theta_hat - q_lo*]
# Reuses the bootstrap replicates (auc_diffs) already computed above
# No new resampling
valid = ~np.isnan(auc_diffs)
diffs = auc_diffs[valid]

theta_hat = auc3 - auc0  # point estimate on the full (original) sample, already computed above

q_lo_star, q_hi_star = np.percentile(diffs, [2.5, 97.5])  # alpha/2 and 1-alpha/2 quantiles of the bootstrap replicates

ci_basic_lo = 2 * theta_hat - q_hi_star
ci_basic_hi = 2 * theta_hat - q_lo_star

print(f"Point estimate (theta_hat): {theta_hat:+.4f}")
print(f"Percentile-method 95% CI: ({q_lo_star:+.4f}, {q_hi_star:+.4f})")
print(f"Basic/reflected bootstrap 95% CI: ({ci_basic_lo:+.4f}, {ci_basic_hi:+.4f})")
print(f"Basic CI excludes zero: {ci_basic_lo > 0}")

In [ ]:
# Reverse-causation check: does the BMI used by the model predate each incident case's diagnosis?
incident_keep = (model_data['EC_case']==0) | ((model_data['EC_case']==1) & model_data['incident_flag'].fillna(False))
md_incident_check = model_data[incident_keep & (model_data['EC_case']==1)].copy()
md_incident_check['bmi_baseline_date'] = pd.to_datetime(md_incident_check['bmi_baseline_date'])
md_incident_check['first_diagnosis_date'] = pd.to_datetime(md_incident_check['first_diagnosis_date'])

md_incident_check['bmi_postdates_diagnosis'] = md_incident_check['bmi_baseline_date'] > md_incident_check['first_diagnosis_date']
md_incident_check['days_bmi_after_diagnosis'] = (
    md_incident_check['bmi_baseline_date'] - md_incident_check['first_diagnosis_date']
).dt.days

n_total = len(md_incident_check)
n_missing_bmi = md_incident_check['bmi_baseline_date'].isna().sum()
n_postdates = md_incident_check['bmi_postdates_diagnosis'].sum()

# AoU small-cell disclosure (<20) applies to missing+postdates combined, since printing the "precedes or on" count/total would let that combined group be reconstructed by subtraction
SMALL_CELL_THRESHOLD = 20
n_precedes_or_on = n_total - n_missing_bmi - n_postdates
n_small_cell_combined = n_missing_bmi + n_postdates

print(f"Incident cases checked: {n_total}")
if n_small_cell_combined < SMALL_CELL_THRESHOLD:
    print(f"  Missing baseline BMI or BMI postdating diagnosis: <{SMALL_CELL_THRESHOLD} cases (suppressed per AoU small-cell policy)")
else:
    print(f"  Missing a baseline BMI measurement entirely: {n_missing_bmi:,} ({100*n_missing_bmi/n_total:.1f}%)")
    print(f"  Selected BMI measurement POSTDATES diagnosis: {n_postdates:,} ({100*n_postdates/n_total:.1f}%)")
    print(f"  Selected BMI measurement precedes or is on diagnosis date: {n_precedes_or_on:,} "
          f"({100*n_precedes_or_on/n_total:.1f}%)")
    if n_postdates >= SMALL_CELL_THRESHOLD:
        delays = md_incident_check.loc[md_incident_check['bmi_postdates_diagnosis'], 'days_bmi_after_diagnosis']
        print(f"\n  Among cases where the model's BMI measurement postdates diagnosis, delay (days): "
              f"min={delays.min():.0f}, median={delays.median():.0f}, max={delays.max():.0f}")

## Age-reference sensitivity check

In [ ]:
# Merge in age at enrolment for the age-reference sensitivity check below
# Only merges if not already present
# Avoids rebuilding model_data from scratch and silently discarding age_z/bmi_z or other columns added by earlier cells in this session.
result = subprocess.run(['gsutil','-u',project_id,'cp',
    f'{bucket_output}/03_ec_cohort_with_baseline_chars.csv', '/home/jupyter/cohort_with_baseline.csv'],
    capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"gsutil cp failed (exit {result.returncode}): {result.stderr}")
cohort_bmi = pd.read_csv('/home/jupyter/cohort_with_baseline.csv', low_memory=False)

if 'age_at_enrolment' not in model_data.columns:
    enrol_cols = cohort_bmi[['person_id', 'age_at_enrolment']].rename(columns={'person_id': '#IID'})
    model_data = model_data.merge(enrol_cols, on='#IID', how='left')

print(f"N: {len(model_data):,}")
print(f"Missing age_at_enrolment: {model_data['age_at_enrolment'].isna().sum():,}")
print("age_at_enrolment in model_data columns:", 'age_at_enrolment' in model_data.columns)
print("age_z in model_data columns:", 'age_z' in model_data.columns)
print("bmi_z in model_data columns:", 'bmi_z' in model_data.columns)

In [ ]:
# Age-reference sensitivity check: age at enrolment vs age at diagnosis, restricted to incident cases. 
# Age-at-enrolment is only a clean common reference time for INCIDENT cases, where enrolment precedes diagnosis (mirroring Wang et al., whose cohort is incident-only by design)
# For prevalent cases it can postdate diagnosis by years, introducing a new reverse-causation problem
# This check does not apply to the full cohort, and the decision to retain prevalent cases in the primary analysis stands
ctrl_enr = model_data[model_data['EC_case']==0]
age_enr_m, age_enr_s = ctrl_enr['age_at_enrolment'].mean(), ctrl_enr['age_at_enrolment'].std()
model_data['age_enrol_z'] = (model_data['age_at_enrolment'] - age_enr_m) / age_enr_s

incident_keep = (model_data['EC_case']==0) | ((model_data['EC_case']==1) & model_data['incident_flag'].fillna(False))
md_incident = model_data[incident_keep].copy()
print(f"Incident-only subset: N={len(md_incident):,}, cases={int(md_incident['EC_case'].sum()):,}")

# Comparator: same incident-only subset, main-text age definition (age at diagnosis for cases, fixed reference date for controls)
d3i = md_incident.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols)
X3i = sm.add_constant(d3i[['PRS_z', 'age_z', 'bmi_z'] + pc_cols])
y3i = d3i['EC_case']
m3i = sm.Logit(y3i, X3i).fit(disp=0)
auc3i = roc_auc_score(y3i, m3i.predict(X3i))

# The actual check: same subset, using age-at-enrolment (common reference time) instead
d3ei = md_incident.dropna(subset=['PRS_z', 'age_enrol_z', 'bmi_z'] + pc_cols)
X3ei = sm.add_constant(d3ei[['PRS_z', 'age_enrol_z', 'bmi_z'] + pc_cols])
y3ei = d3ei['EC_case']
m3ei = sm.Logit(y3ei, X3ei).fit(disp=0)
auc3ei = roc_auc_score(y3ei, m3ei.predict(X3ei))

report(m3i, X3i, y3i, auc3i, "INCIDENT-ONLY, age-at-diagnosis (main-text definition): PRS + age + BMI + 10 PCs")
report(m3ei, X3ei, y3ei, auc3ei, "INCIDENT-ONLY, age-at-enrolment (common ref. time): PRS + age + BMI + 10 PCs")

print(f"\n=== Age-reference comparison, incident-only subset (N={len(y3i):,}, cases={int(y3i.sum()):,}) ===")
print(f"Age-at-diagnosis:   PRS OR {np.exp(m3i.params['PRS_z']):.4f}, age OR {np.exp(m3i.params['age_z']):.4f}, "
      f"BMI OR {np.exp(m3i.params['bmi_z']):.4f}, AUC {auc3i:.4f}")
print(f"Age-at-enrolment:   PRS OR {np.exp(m3ei.params['PRS_z']):.4f}, age OR {np.exp(m3ei.params['age_enrol_z']):.4f}, "
      f"BMI OR {np.exp(m3ei.params['bmi_z']):.4f}, AUC {auc3ei:.4f}")

In [ ]:
# Confirming the two age-reference specifications use the same participant set.
ids_d3i = set(d3i['#IID']) if '#IID' in d3i.columns else set(d3i.index)
ids_d3ei = set(d3ei['#IID']) if '#IID' in d3ei.columns else set(d3ei.index)
assert ids_d3i == ids_d3ei, (
    f"d3i and d3ei use different participant sets ({len(ids_d3i)} vs {len(ids_d3ei)}); "
    f"the age-reference comparison is not a clean paired comparison until this is resolved."
)
print(f"Confirmed: d3i (age-at-diagnosis) and d3ei (age-at-enrolment) use identical participant sets, N={len(ids_d3i):,}.")

## PRS percentile-bin adjustment-level comparison

In [ ]:
# Top-1%-vs-middle-quintile contrast at three adjustment levels
# Checks how much of the percentile-bin PRS-OR model's adjusted tail contrast survives without PC adjustment
# Uses df/model/bin_order from the percentile-bin table above
top1_mask = df['prs_bin'] == 'Top 1%'
mid_mask = df['prs_bin'] == 'Middle quintile (40-60%)'
top1_cases = df.loc[top1_mask, 'EC_case'].sum(); top1_n = top1_mask.sum()
mid_cases = df.loc[mid_mask, 'EC_case'].sum(); mid_n = mid_mask.sum()
crude_or = (top1_cases / (top1_n - top1_cases)) / (mid_cases / (mid_n - mid_cases))
print(f"(1) Crude OR (top 1% vs middle quintile, no adjustment): {crude_or:.3f}")

# PC-adjusted read directly from the existing, already-fitted 7-bin model
# Reproduces the percentile-bin model's OR exactly; no refitting, no separation risk
or_pc = np.exp(model.params['Top 1%'])
ci_pc = np.exp(model.conf_int().loc['Top 1%'])
print(f"(2) PC-adjusted OR (10 PCs, no ancestry, percentile-bin model value): {or_pc:.3f} (95% CI {ci_pc[0]:.3f}-{ci_pc[1]:.3f})")

# PC + ancestry-adjusted
# Same full 7-bin design, full sample, ancestry dummies added
anc_dummies = pd.get_dummies(df['ancestry_pred'], prefix='anc', drop_first=True).astype(float)
X_anc = sm.add_constant(pd.concat([
    pd.get_dummies(df['prs_bin'], drop_first=False).drop(columns=['Middle quintile (40-60%)']).astype(float),
    df[pc_cols],
    anc_dummies
], axis=1))
model_anc = sm.Logit(y, X_anc).fit(disp=0)
or_pc_anc = np.exp(model_anc.params['Top 1%'])
ci_pc_anc = np.exp(model_anc.conf_int().loc['Top 1%'])
print(f"(3) PC + ancestry-adjusted OR: {or_pc_anc:.3f} (95% CI {ci_pc_anc[0]:.3f}-{ci_pc_anc[1]:.3f})")

print(f"\n=== Summary: top 1% vs middle quintile contrast, by adjustment level ===")
print(f"Crude (no adjustment):        {crude_or:.3f}")
print(f"PC-adjusted (percentile-bin model): {or_pc:.3f}")
print(f"PC + ancestry-adjusted:       {or_pc_anc:.3f}")

## Cross-validated AUC and calibration

In [ ]:
# 5-fold stratified cross-validation
# Held-out AUC for the full model vs the no-PRS comparator as the counterpart to the apparent-performance bootstrap above.
from sklearn.model_selection import StratifiedKFold

d_full_cv = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols).reset_index(drop=True)
X_full_all = d_full_cv[['PRS_z', 'age_z', 'bmi_z'] + pc_cols]
X_noprs_all = d_full_cv[['age_z', 'bmi_z'] + pc_cols]
y_all_cv = d_full_cv['EC_case']

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_auc_full = []
cv_auc_noprs = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full_all, y_all_cv)):
    y_train, y_test = y_all_cv.iloc[train_idx], y_all_cv.iloc[test_idx]

    X_train_full = sm.add_constant(X_full_all.iloc[train_idx])
    X_test_full = sm.add_constant(X_full_all.iloc[test_idx], has_constant='add')
    X_test_full = X_test_full[X_train_full.columns]
    m_full = sm.Logit(y_train, X_train_full).fit(disp=0)
    pred_full = m_full.predict(X_test_full)
    cv_auc_full.append(roc_auc_score(y_test, pred_full))

    X_train_noprs = sm.add_constant(X_noprs_all.iloc[train_idx])
    X_test_noprs = sm.add_constant(X_noprs_all.iloc[test_idx], has_constant='add')
    X_test_noprs = X_test_noprs[X_train_noprs.columns]
    m_noprs = sm.Logit(y_train, X_train_noprs).fit(disp=0)
    pred_noprs = m_noprs.predict(X_test_noprs)
    cv_auc_noprs.append(roc_auc_score(y_test, pred_noprs))

    print(f"Fold {fold+1}: AUC full={cv_auc_full[-1]:.4f}, AUC no-PRS={cv_auc_noprs[-1]:.4f}, "
          f"gain={cv_auc_full[-1]-cv_auc_noprs[-1]:+.4f}")

cv_auc_full = np.array(cv_auc_full)
cv_auc_noprs = np.array(cv_auc_noprs)
cv_gain = cv_auc_full - cv_auc_noprs

print(f"\n=== {n_splits}-fold cross-validated AUC ===")
print(f"Full model (PRS+age+BMI+PCs):  mean AUC = {cv_auc_full.mean():.4f} (SD {cv_auc_full.std():.4f})")
print(f"No-PRS model (age+BMI+PCs):    mean AUC = {cv_auc_noprs.mean():.4f} (SD {cv_auc_noprs.std():.4f})")
print(f"Mean CV AUC gain from adding PRS: {cv_gain.mean():+.4f} (SD {cv_gain.std():.4f})")
print(f"Gain positive in {int((cv_gain > 0).sum())}/{n_splits} folds")

In [ ]:
# Figure: incremental AUC from adding PRS
# Apparent-performance bootstrap (Panel A) vs held-out cross-validation (Panel B)
# Reuses auc_diffs (1000 bootstrap replicates) and cv_gain (5 CV folds) from the two cells above
fig, axes = plt.subplots(2, 1, figsize=(6.0, 6.0))

# Panel A: bootstrap distribution
ax = axes[0]
ax.hist(auc_diffs, bins=40, color=GREY, alpha=0.55, edgecolor='none')
ax.axvline(0, color='#888888', linestyle=':', linewidth=1)
theta_hat_plot = auc3 - auc0
ax.axvline(theta_hat_plot, color=NAVY, linewidth=2)
q_lo_plot, q_hi_plot = np.percentile(auc_diffs, [2.5, 97.5])
ax.axvspan(q_lo_plot, q_hi_plot, color=NAVY, alpha=0.12)
ax.text(theta_hat_plot, ax.get_ylim()[1]*0.92, f'  +{theta_hat_plot:.4f}', color=NAVY, fontsize=9, va='top')
ax.set_xlabel('AUC gain')
ax.set_ylabel('Count')
ax.set_title('A. Apparent (bootstrap)', loc='left', fontsize=11, fontweight='bold')
style_ax(ax)

# Panel B: 5-fold cross-validation
ax = axes[1]
fold_x = np.arange(1, len(cv_gain) + 1)
colors_cv = [NAVY if g > 0 else CORAL for g in cv_gain]
ax.scatter(fold_x, cv_gain, c=colors_cv, s=70, zorder=3)
ax.axhline(0, color='#888888', linestyle=':', linewidth=1)
ax.axhline(cv_gain.mean(), color=NAVY, linestyle='--', linewidth=1.3)
ax.text(len(cv_gain) + 0.15, cv_gain.mean(), f'mean {cv_gain.mean():+.4f}', color=NAVY,
        fontsize=9, va='bottom')
ax.set_xticks(fold_x)
ax.set_xlabel('Cross-validation fold')
ax.set_ylabel('AUC gain (held-out)')
ax.set_xlim(0.5, len(cv_gain) + 1.3)
ax.set_title('B. Held-out (cross-validation)', loc='left', fontsize=11, fontweight='bold')
style_ax(ax)

plt.tight_layout()
savefig_both(fig, 'prs_incremental_auc_validation', project_id, bucket_output)
plt.show()

In [ ]:
# Out-of-fold calibration slope/intercept using the same 5-fold split as above
# Change is what we store each participant's prediction from the fold where they were held out
from scipy.special import logit as sp_logit

d_full_cv = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols).reset_index(drop=True)
X_full_all = d_full_cv[['PRS_z', 'age_z', 'bmi_z'] + pc_cols]
y_all_cv = d_full_cv['EC_case']

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred = np.full(len(d_full_cv), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_full_all, y_all_cv)):
    y_train = y_all_cv.iloc[train_idx]
    X_train_full = sm.add_constant(X_full_all.iloc[train_idx])
    X_test_full = sm.add_constant(X_full_all.iloc[test_idx], has_constant='add')
    X_test_full = X_test_full[X_train_full.columns]
    m_full = sm.Logit(y_train, X_train_full).fit(disp=0)
    oof_pred[test_idx] = m_full.predict(X_test_full)

assert not np.isnan(oof_pred).any(), "every row should get exactly one out-of-fold prediction"

# Calibration-slope regression on pooled out-of-fold predictions
oof_logit = sp_logit(np.clip(oof_pred, 1e-10, 1 - 1e-10))
X_cal = sm.add_constant(pd.DataFrame({'oof_logit': oof_logit}))
m_cal = sm.Logit(y_all_cv.reset_index(drop=True), X_cal).fit(disp=0)

print(f"Out-of-fold calibration slope:     {m_cal.params['oof_logit']:.4f}")
print(f"Out-of-fold calibration intercept: {m_cal.params['const']:.4f}")
print(f"N = {len(d_full_cv):,}")

# Decile comparison on out-of-fold predictions
d_oof = d_full_cv.copy()
d_oof['oof_pred'] = oof_pred
d_oof['oof_decile'] = pd.qcut(d_oof['oof_pred'], 10, labels=False, duplicates='drop')
oof_calib_table = d_oof.groupby('oof_decile').agg(
    n=('EC_case', 'count'),
    observed_cases=('EC_case', 'sum'),
    mean_predicted_risk=('oof_pred', 'mean')
).reset_index()
oof_calib_table['observed_risk'] = oof_calib_table['observed_cases'] / oof_calib_table['n']
oof_calib_table['obs_minus_exp'] = oof_calib_table['observed_risk'] - oof_calib_table['mean_predicted_risk']
print(oof_calib_table.to_string(index=False))

In [ ]:
# In-sample calibration (slope/intercept + decile table)
# For comparison against the out-of-fold version above and below
d_calib = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols).reset_index(drop=True)
X_calib = sm.add_constant(d_calib[['PRS_z', 'age_z', 'bmi_z'] + pc_cols])
y_calib = d_calib['EC_case']

m_calib = sm.Logit(y_calib, X_calib).fit(disp=0)
d_calib['predicted_risk'] = m_calib.predict(X_calib)

from scipy.special import logit as sp_logit
d_calib['predicted_logit'] = sp_logit(d_calib['predicted_risk'].clip(1e-10, 1-1e-10))
X_cal_slope = sm.add_constant(d_calib[['predicted_logit']])
m_cal_slope = sm.Logit(y_calib, X_cal_slope).fit(disp=0)
calib_slope = m_cal_slope.params['predicted_logit']
print(f"Calibration slope (ideal = 1.0): {calib_slope:.3f}")
print(f"Calibration intercept (ideal = 0.0): {m_cal_slope.params['const']:.3f}")

d_calib['risk_decile'] = pd.qcut(d_calib['predicted_risk'], 10, labels=False, duplicates='drop')
calib_table = d_calib.groupby('risk_decile').agg(
    n=('EC_case', 'count'),
    observed_cases=('EC_case', 'sum'),
    mean_predicted_risk=('predicted_risk', 'mean')
).reset_index()
calib_table['observed_risk'] = calib_table['observed_cases'] / calib_table['n']
calib_table['obs_minus_exp'] = calib_table['observed_risk'] - calib_table['mean_predicted_risk']

print("\n=== Calibration by decile of predicted risk ===")
print(calib_table.to_string(index=False))

## GLM diagnostics

In [ ]:
# GLM residual diagnostics (standardised Pearson/deviance residuals, leverage, Cook's distance) for the pooled full model
# Standard logistic-regression diagnostic practice is followed
# N is too large for per-point plots, so this reports summary statistics and threshold-exceedance counts instead.
d_diag = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols).reset_index(drop=True)
X_diag = sm.add_constant(d_diag[['PRS_z', 'age_z', 'bmi_z'] + pc_cols])
y_diag = d_diag['EC_case']

m_diag = sm.Logit(y_diag, X_diag).fit(disp=0)
mu_hat = m_diag.predict(X_diag)
n_diag, p_diag = X_diag.shape

# Pearson residuals: r_Pi = (y_i - mu_i) / sqrt(mu_i(1-mu_i))   [phi=1 for Bernoulli]
V_mu = mu_hat * (1 - mu_hat)
pearson_resid = (y_diag - mu_hat) / np.sqrt(V_mu)

# Deviance residuals: sign(y_i - mu_i) * sqrt(d_i), Bernoulli case
eps = 1e-10
mu_clip = mu_hat.clip(eps, 1 - eps)
d_i = -2 * (y_diag * np.log(mu_clip) + (1 - y_diag) * np.log(1 - mu_clip))
deviance_resid = np.sign(y_diag - mu_hat) * np.sqrt(d_i.clip(lower=0))

# Leverage h_ii = diagonal of H = W^(1/2) X (X'WX)^-1 X' W^(1/2), computed row-wise without forming the full n x n matrix (infeasible at this N)
W_diag = V_mu.values
XtWX_inv = np.linalg.inv((X_diag.values * W_diag[:, None]).T @ X_diag.values)
# h_ii = w_i * x_i' (X'WX)^-1 x_i
Xv = X_diag.values
h_ii = W_diag * np.einsum('ij,jk,ik->i', Xv, XtWX_inv, Xv)

# Standardised residuals
pearson_std = pearson_resid / np.sqrt(1 - h_ii)
deviance_std = deviance_resid / np.sqrt(1 - h_ii)

# Cook's distance: C_i = (beta_hat - beta_hat_-i)' (X'WX) (beta_hat - beta_hat_-i) / p -- approximated via the standard one-step formula C_i = r_Pi^2 * h_ii / (p * (1 - h_ii))
cooks_d = (pearson_std**2) * h_ii / (p_diag * (1 - h_ii))

print(f"=== GLM Diagnostics Summary (N={n_diag:,}, p={p_diag}) ===\n")

print("--- Standardised Pearson residuals ---")
print(f"Mean: {pearson_std.mean():.4f}, SD: {pearson_std.std():.4f}")
print(f"% with |r'_P| > 2: {100*(np.abs(pearson_std) > 2).mean():.2f}%")
print(f"% with |r'_P| > 3: {100*(np.abs(pearson_std) > 3).mean():.2f}%")

print("\n--- Standardised deviance residuals ---")
print(f"Mean: {deviance_std.mean():.4f}, SD: {deviance_std.std():.4f}")
print(f"% with |r'_D| > 2: {100*(np.abs(deviance_std) > 2).mean():.2f}%")
print(f"% with |r'_D| > 3: {100*(np.abs(deviance_std) > 3).mean():.2f}%")

print("\n--- Leverage ---")
avg_leverage = p_diag / n_diag
leverage_threshold = 2 * avg_leverage
print(f"Average leverage (p/n): {avg_leverage:.6f}")
print(f"High-leverage threshold (2p/n): {leverage_threshold:.6f}")
print(f"N exceeding 2p/n: {(h_ii > leverage_threshold).sum():,} ({100*(h_ii > leverage_threshold).mean():.2f}%)")
print(f"Max leverage: {h_ii.max():.6f}")

print("\n--- Cook's distance ---")
cooks_threshold = 8 / (n_diag - 2 * p_diag)
print(f"High-influence threshold (8/(n-2p)): {cooks_threshold:.8f}")
print(f"N exceeding threshold: {(cooks_d > cooks_threshold).sum():,} ({100*(cooks_d > cooks_threshold).mean():.2f}%)")
print(f"Max Cook's distance: {cooks_d.max():.8f}")

# Pearson goodness-of-fit statistic
# Not chi-sq distributed here, since this is Bernoulli data with m_i=1, which invalidates that approximation
X2 = (pearson_resid**2).sum()
print(f"\n--- Pearson goodness-of-fit statistic (context only) ---")
print(f"X^2 = {X2:.1f} (NOT referred to chi-sq(n-p): invalid for Bernoulli/binary data)")

## BMI as confounder vs mediator

In [ ]:
# BMI confounder-vs-mediator check: decomposing the PRS OR attenuation across PCs -> +age -> +age+BMI to see whether BMI specifically drives the attenuation
# This is consistent with BMI partially mediating the PRS-EC association
d_bmi_check = model_data.dropna(subset=['PRS_z', 'age_z', 'bmi_z'] + pc_cols).reset_index(drop=True)
y_bmi_check = d_bmi_check['EC_case']

specs = {
    'PRS + PCs only':        ['PRS_z'] + pc_cols,
    '+ age (no BMI)':        ['PRS_z', 'age_z'] + pc_cols,
    '+ BMI (no age)':        ['PRS_z', 'bmi_z'] + pc_cols,
    '+ age + BMI (full)':    ['PRS_z', 'age_z', 'bmi_z'] + pc_cols,
}

print(f"=== BMI vs age attenuation decomposition (N={len(d_bmi_check):,}) ===\n")
results = {}
for name, cols in specs.items():
    X_spec = sm.add_constant(d_bmi_check[cols])
    m_spec = sm.Logit(y_bmi_check, X_spec).fit(disp=0)
    or_prs = np.exp(m_spec.params['PRS_z'])
    ci_prs = np.exp(m_spec.conf_int().loc['PRS_z'])
    results[name] = or_prs
    print(f"{name:25s}: PRS OR = {or_prs:.4f}  (95% CI {ci_prs[0]:.4f}-{ci_prs[1]:.4f})")

base = results['PRS + PCs only']
print(f"\n--- Attenuation relative to PRS+PCs baseline (OR={base:.4f}) ---")
for name in ['+ age (no BMI)', '+ BMI (no age)', '+ age + BMI (full)']:
    pct_drop = 100 * (base - results[name]) / base
    print(f"{name:25s}: {pct_drop:+.2f}% change in OR")

In [ ]:
# PRS -> BMI mediation-mechanism check: tests the first link of a potential PRS -> BMI -> EC pathway
# If PRS does not predict BMI, a mediating role for BMI is implausible, regardless of the attenuation pattern above
# Restricted to controls to avoid reverse causation from disease-related BMI change
d_mediation = model_data.loc[model_data['EC_case'] == 0].dropna(subset=['PRS_z', 'bmi_z'] + pc_cols).reset_index(drop=True)

X_med = sm.add_constant(d_mediation[['PRS_z'] + pc_cols])
y_med = d_mediation['bmi_z']  # continuous outcome -> OLS, not logistic

m_med = sm.OLS(y_med, X_med).fit()

print(f"=== PRS -> BMI association among controls (N={len(d_mediation):,}) ===\n")
print(f"PRS_z coefficient on BMI_z: {m_med.params['PRS_z']:.5f}")
print(f"Standard error:             {m_med.bse['PRS_z']:.5f}")
print(f"t-statistic:                {m_med.tvalues['PRS_z']:.3f}")
print(f"p-value:                    {m_med.pvalues['PRS_z']:.4f}")
ci_med = m_med.conf_int().loc['PRS_z']
print(f"95% CI:                     ({ci_med[0]:.5f}, {ci_med[1]:.5f})")
print(f"\nModel R-squared: {m_med.rsquared:.5f}")

In [ ]:
# Out-of-fold calibration plot (companion to the in-sample plot above)
# Uses the oof_pred array computed in the out-of-fold calibration cell above
import matplotlib.pyplot as plt

d_calib_oof = d_full_cv.copy()
d_calib_oof['oof_pred'] = oof_pred
d_calib_oof['oof_decile'] = pd.qcut(d_calib_oof['oof_pred'], 10, labels=False, duplicates='drop')

calib_table_oof = d_calib_oof.groupby('oof_decile').agg(
    n=('EC_case', 'count'),
    observed_cases=('EC_case', 'sum'),
    mean_predicted_risk=('oof_pred', 'mean')
).reset_index()
calib_table_oof['observed_risk'] = calib_table_oof['observed_cases'] / calib_table_oof['n']
calib_table_oof['obs_minus_exp'] = calib_table_oof['observed_risk'] - calib_table_oof['mean_predicted_risk']

print("=== Out-of-fold calibration by decile of predicted risk ===")
print(calib_table_oof.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.3, 4.3))
ax.plot(calib_table_oof['mean_predicted_risk'], calib_table_oof['observed_risk'],
        'o-', color=NAVY, markersize=6, linewidth=1.6, label='Observed (out-of-fold)')
lims = [0, max(calib_table_oof['mean_predicted_risk'].max(), calib_table_oof['observed_risk'].max()) * 1.1]
ax.plot(lims, lims, '--', color=GREY, linewidth=1.2)
# Direct label at the end of the identity line instead of a separate legend entry
ax.text(lims[1], lims[1], '  Perfect calibration', color='#777777', fontsize=8.5,
        va='center', ha='left', rotation=0)
ax.set_xlabel('Mean predicted risk (decile)')
ax.set_ylabel('Observed risk (decile)')
ax.set_xlim(lims[0], lims[1] * 1.28)
style_ax(ax)
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
savefig_both(fig, 'calibration_plot_oof', project_id, bucket_output)
plt.show()

In [ ]:
# Incremental R^2: PRS's own contribution to BMI, beyond the 10 PCs
# Reuses d_mediation from the PRS -> BMI mediation-mechanism check above
r2_full = m_med.rsquared  # BMI_z ~ PRS_z + 10 PCs (already fitted as m_med)

X_med_reduced = sm.add_constant(d_mediation[pc_cols])  # BMI_z ~ 10 PCs only
m_med_reduced = sm.OLS(y_med, X_med_reduced).fit()
r2_reduced = m_med_reduced.rsquared

incremental_r2 = r2_full - r2_reduced

print(f"=== Incremental R^2 of PRS on BMI, beyond 10 PCs (N={len(d_mediation):,}) ===\n")
print(f"Full model R^2 (PRS + 10 PCs):      {r2_full:.5f}")
print(f"Reduced model R^2 (10 PCs only):    {r2_reduced:.5f}")
print(f"Incremental R^2 from PRS alone:     {incremental_r2:.5f}")


## PRS x ancestry interaction test

In [ ]:
# Formal likelihood-ratio test of PRS x ancestry interaction
# Restricted to the five well-powered ancestry groups used throughout the ancestry-stratified analysis
well_powered = ['eur', 'afr', 'amr', 'eas', 'oth']
d_interact = model_data.loc[model_data['ancestry_pred'].isin(well_powered)].dropna(
    subset=['PRS_z'] + pc_cols
).reset_index(drop=True)

d_interact['ancestry_pred'] = pd.Categorical(
    d_interact['ancestry_pred'], categories=well_powered
)
anc_dummies_int = pd.get_dummies(d_interact['ancestry_pred'], prefix='anc', drop_first=True).astype(float)

y_int = d_interact['EC_case'].reset_index(drop=True)

# Model 1: main effects only (PRS + PCs + ancestry, no interaction)
X_main = sm.add_constant(pd.concat([
    d_interact[['PRS_z'] + pc_cols].reset_index(drop=True),
    anc_dummies_int.reset_index(drop=True)
], axis=1))
m_main = sm.Logit(y_int, X_main).fit(disp=0)

# Model 2: main effects + PRS x ancestry interaction terms
interaction_terms = anc_dummies_int.mul(d_interact['PRS_z'].reset_index(drop=True), axis=0)
interaction_terms.columns = [f'PRS_x_{c}' for c in interaction_terms.columns]

X_interact = sm.add_constant(pd.concat([
    d_interact[['PRS_z'] + pc_cols].reset_index(drop=True),
    anc_dummies_int.reset_index(drop=True),
    interaction_terms.reset_index(drop=True)
], axis=1))
m_interact = sm.Logit(y_int, X_interact).fit(disp=0)

# Likelihood ratio test: does adding PRS x ancestry interactions improve fit?
lr_stat = 2 * (m_interact.llf - m_main.llf)
df_diff = m_interact.df_model - m_main.df_model
from scipy.stats import chi2
lr_pvalue = chi2.sf(lr_stat, df_diff)

print(f"=== Likelihood ratio test: PRS x ancestry interaction ===\n")
print(f"N = {len(d_interact):,}, well-powered ancestry groups only (eur/afr/amr/eas/oth)\n")
print(f"Main-effects-only model log-likelihood:      {m_main.llf:.3f}")
print(f"Main-effects + interaction log-likelihood:   {m_interact.llf:.3f}")
print(f"LR statistic:                                {lr_stat:.3f}")
print(f"df (number of interaction terms):            {df_diff}")
print(f"p-value:                                     {lr_pvalue:.4g}")
print(f"\nNote: a non-significant p-value does not prove the associations are equal, only")
print(f"that this test does not detect heterogeneity at this sample size.")

print(f"\n=== Interaction term coefficients (PRS effect modification by ancestry, vs European) ===")
for term in interaction_terms.columns:
    coef = m_interact.params[term]
    ci = m_interact.conf_int().loc[term]
    pval = m_interact.pvalues[term]
    print(f"{term}: coef={coef:.4f} (OR multiplier={np.exp(coef):.3f}), "
          f"95% CI=({np.exp(ci[0]):.3f}, {np.exp(ci[1]):.3f}), p={pval:.4f}")

## Fold-level standardisation-reference check

In [ ]:
# Verification: does the standardisation reference (full-cohort controls vs training-fold-only controls) change predicted probabilities? 
# Runs on fold 1 of the existing 5-fold split, using real cohort data.
skf_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(iter(skf_check.split(X_full_all, y_all_cv)))

# Version A: standardisation as actually used (controls from full cohort)
# PRS_z, age_z, bmi_z are already the full-cohort-control-standardised columns
X_train_A = sm.add_constant(X_full_all.iloc[train_idx])
X_test_A = sm.add_constant(X_full_all.iloc[test_idx], has_constant='add')
X_test_A = X_test_A[X_train_A.columns]
y_train_A = y_all_cv.iloc[train_idx]
m_A = sm.Logit(y_train_A, X_train_A).fit(disp=0)
pred_A = m_A.predict(X_test_A)

# Version B: re-standardise age_z, bmi_z, PRS_z using controls from the training fold only
# Then refit and predict on the test fold
train_data = d_full_cv.iloc[train_idx].copy()
test_data = d_full_cv.iloc[test_idx].copy()

train_controls = train_data[train_data['EC_case'] == 0]

# Recompute standardisation constants from training-fold controls only
prs_m_fold, prs_s_fold = train_controls['PRS_raw'].mean(), train_controls['PRS_raw'].std()
age_m_fold, age_s_fold = train_controls['age_display'].mean(), train_controls['age_display'].std()
bmi_m_fold, bmi_s_fold = train_controls['bmi_baseline'].mean(), train_controls['bmi_baseline'].std()

for _fold_df in (train_data, test_data):
    _fold_df['PRS_z_fold'] = (_fold_df['PRS_raw'] - prs_m_fold) / prs_s_fold
    _fold_df['age_z_fold'] = (_fold_df['age_display'] - age_m_fold) / age_s_fold
    _fold_df['bmi_z_fold'] = (_fold_df['bmi_baseline'] - bmi_m_fold) / bmi_s_fold

fold_cols = ['PRS_z_fold', 'age_z_fold', 'bmi_z_fold'] + pc_cols
X_train_B = sm.add_constant(train_data[fold_cols])
X_test_B = sm.add_constant(test_data[fold_cols], has_constant='add')
X_test_B = X_test_B[X_train_B.columns]
y_train_B = train_data['EC_case']
m_B = sm.Logit(y_train_B, X_train_B).fit(disp=0)
pred_B = m_B.predict(X_test_B)

# Compare
max_diff = np.max(np.abs(pred_A.values - pred_B.values))
print(f"Max absolute difference in predicted probabilities (fold 1, N={len(test_idx):,}): {max_diff:.2e}")
print(f"Predictions identical to within 1e-8: {np.allclose(pred_A.values, pred_B.values, atol=1e-8)}")

## Functional-form (spline) check

In [ ]:
# Functional-form check: is the log-odds of EC linear in PRS_z?
# The PRS_z term in the baseline PRS+PCs model is replaced by a natural cubic
# spline (3 knots) 
# It is then compared to the linear specification by a likelihood-ratio test. 
# The spline basis is built without patsy's own intercept column("- 1"), with one basis column dropped manually before adding sm's constant
# This is because the 3 spline basis functions otherwise sum to a constant (collinear with any intercept term)
# A rank check confirms this is resolved.
import patsy

d_spline = model_data.dropna(subset=['PRS_z'] + pc_cols).reset_index(drop=True)
y_spline = d_spline['EC_case']

# Linear model
X_linear = sm.add_constant(d_spline[['PRS_z'] + pc_cols])
m_linear = sm.Logit(y_spline, X_linear).fit(disp=0)

# Natural cubic spline basis, no patsy-generated intercept, then drop one column manually to remove the remaining linear dependence
basis_full = patsy.dmatrix("cr(PRS_z, df=3) - 1", d_spline, return_type='dataframe')
spline_design_info = basis_full.design_info  # keep for correct prediction later
basis = basis_full.iloc[:, 1:]  # drop one redundant column

X_spline = sm.add_constant(pd.concat([
    basis.reset_index(drop=True),
    d_spline[pc_cols].reset_index(drop=True)
], axis=1), has_constant='add')

rank = np.linalg.matrix_rank(X_spline.to_numpy())
print(f"X_spline: {X_spline.shape[1]} columns, rank {rank} -- full rank: {rank == X_spline.shape[1]}")
assert rank == X_spline.shape[1], "X_spline is still rank-deficient"

m_spline = sm.Logit(y_spline, X_spline).fit(disp=0)

lr_stat = 2 * (m_spline.llf - m_linear.llf)
df_diff = m_spline.df_model - m_linear.df_model
from scipy.stats import chi2
lr_p = chi2.sf(lr_stat, df_diff)

print(f"Linear model log-likelihood:  {m_linear.llf:.3f}")
print(f"Spline model log-likelihood:  {m_spline.llf:.3f}")
print(f"LR statistic: {lr_stat:.3f} on {df_diff} df, p = {lr_p:.4f}")
print(f"\nLinear PRS_z OR/SD: {np.exp(m_linear.params['PRS_z']):.4f}")

# Prediction: reusing the fitted model's basis (same column-dropping convention)
import matplotlib.pyplot as plt

prs_grid = np.linspace(d_spline['PRS_z'].quantile(0.01), d_spline['PRS_z'].quantile(0.99), 200)
pc_means = d_spline[pc_cols].mean()

X_grid_linear = sm.add_constant(pd.DataFrame({'PRS_z': prs_grid, **{c: pc_means[c] for c in pc_cols}}), has_constant='add')
X_grid_linear = X_grid_linear[X_linear.columns]
logodds_linear = X_grid_linear @ m_linear.params

grid_basis_full = patsy.build_design_matrices([spline_design_info], {"PRS_z": prs_grid}, return_type='dataframe')[0]
spline_grid_basis = grid_basis_full.iloc[:, 1:]  # same dropped-column convention as training

X_grid_spline = sm.add_constant(pd.concat([
    spline_grid_basis.reset_index(drop=True),
    pd.DataFrame({c: [pc_means[c]]*len(prs_grid) for c in pc_cols})
], axis=1), has_constant='add')
X_grid_spline = X_grid_spline[X_spline.columns]
logodds_spline = X_grid_spline @ m_spline.params

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.plot(prs_grid, logodds_linear - logodds_linear.mean(), '--', color=GREY,
        linewidth=1.8, label='Linear PRS_z term')
ax.plot(prs_grid, logodds_spline - logodds_spline.mean(), '-', color=NAVY,
        linewidth=2, label='Natural cubic spline (3 knots)')
ax.set_xlabel('PRS_z')
ax.set_ylabel('Fitted log-odds (centred)')
style_ax(ax)
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
savefig_both(fig, 'prs_functional_form_check', project_id, bucket_output)
plt.show()

## Figures: crude vs adjusted percentile-bin contrast

In [ ]:
# Figure: crude vs PC-adjusted OR by PRS percentile bin
# Visualising that the adjusted top-1% contrast is substantially larger than its crude (unadjusted) counterpart
crude_rows = []
mid_mask = df['prs_bin'] == 'Middle quintile (40-60%)'
mid_cases = df.loc[mid_mask, 'EC_case'].sum(); mid_n = mid_mask.sum()

for b in bin_order:
    bin_mask = df['prs_bin'] == b
    if b == 'Middle quintile (40-60%)':
        crude_rows.append({'Bin': b, 'crude_OR': 1.0, 'crude_lo': 1.0, 'crude_hi': 1.0})
        continue
    bin_cases = df.loc[bin_mask, 'EC_case'].sum(); bin_n = bin_mask.sum()
    # 2x2 table: bin vs middle quintile
    a, b_ = bin_cases, bin_n - bin_cases
    c, d_ = mid_cases, mid_n - mid_cases
    crude_or = (a / b_) / (c / d_)
    # Woolf's log-OR CI
    se_log_or = np.sqrt(1/a + 1/b_ + 1/c + 1/d_)
    log_or = np.log(crude_or)
    crude_lo = np.exp(log_or - 1.96*se_log_or)
    crude_hi = np.exp(log_or + 1.96*se_log_or)
    crude_rows.append({'Bin': b, 'crude_OR': crude_or, 'crude_lo': crude_lo, 'crude_hi': crude_hi})

crude_df = pd.DataFrame(crude_rows)
compare_df = decile_table.merge(crude_df, on='Bin')
compare_df['OR'] = pd.to_numeric(compare_df['OR'], errors='coerce').fillna(1.0)
compare_df['CI_low'] = pd.to_numeric(compare_df['CI_low'], errors='coerce').fillna(1.0)
compare_df['CI_high'] = pd.to_numeric(compare_df['CI_high'], errors='coerce').fillna(1.0)

print(compare_df[['Bin','crude_OR','crude_lo','crude_hi','OR','CI_low','CI_high']].to_string(index=False))

fig, ax = plt.subplots(figsize=(5.3, 3.4))
y_pos = np.arange(len(compare_df))[::-1]
offset = 0.17

ax.errorbar(compare_df['crude_OR'], y_pos - offset,
            xerr=[compare_df['crude_OR']-compare_df['crude_lo'], compare_df['crude_hi']-compare_df['crude_OR']],
            fmt='o', color=GREY, ecolor=GREY, capsize=3, markersize=6, label='Crude (unadjusted)')
ax.errorbar(compare_df['OR'], y_pos + offset,
            xerr=[compare_df['OR']-compare_df['CI_low'], compare_df['CI_high']-compare_df['OR']],
            fmt='s', color=NAVY, ecolor=NAVY, capsize=3, markersize=6, label='PC-adjusted')

ax.set_xscale('log')
ax.axvline(1, color=GREY, linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(compare_df['Bin'])
ax.set_xlabel('OR vs middle quintile (log scale)')
style_ax(ax)
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.08), ncol=2)
plt.tight_layout()
savefig_both(fig, 'prs_percentile_crude_vs_adjusted', project_id, bucket_output)
plt.show()